# Encoding techniques and quantum classification

## State reachability, kernel geometry, separator expressivity, and empirical trainability

In this lecture we hold the dataset and downstream learning algorithms fixed while changing the quantum feature map.

We will distinguish three related ideas:

1. **State reachability:** Which quantum states can be produced as the data point varies?
2. **Separator expressivity:** Which decision functions become accessible after the encoded states are processed?
3. **Empirical trainability:** Can a derivative-free optimizer find useful variational parameters reliably?

The objective is to expose how encoding changes the learning problem, no advantage claims here!

In [ ]:
# %pip install -q \
#     "qiskit==2.5.2" \
#     "qiskit-machine-learning==0.9.1" \
#     "kagglehub>=0.3" \
#     "scikit-learn>=1.6" \
#     "pandas>=2.2" \
#     "matplotlib>=3.9" \
#     "seaborn>=0.13" \
#     "pylatexenc>=2.10"

In [ ]:
from pathlib import Path

import kagglehub
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from IPython.display import display

from sklearn.base import clone
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import ConfusionMatrixDisplay, classification_report
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, MinMaxScaler, StandardScaler
from sklearn.svm import SVC

from qiskit import QuantumCircuit, transpile
from qiskit.circuit import ParameterVector
from qiskit.circuit.library import efficient_su2, zz_feature_map

from qiskit_machine_learning.algorithms import VQC
from qiskit_machine_learning.circuit.library import raw_feature_vector
from qiskit_machine_learning.kernels import FidelityStatevectorKernel
from qiskit_machine_learning.optimizers import COBYLA
from qiskit_machine_learning.primitives import QMLSampler
from qiskit_machine_learning.utils import algorithm_globals

from qml_encoding_utils import (
    kernel_diagnostics,
    plot_classifier_grid,
    plot_decision_spectra,
    plot_kernel_geometry,
    plot_kernel_matrices,
    plot_loss_histories,
    plot_vqc_boundaries,
    separator_diagnostics,
)

sns.set_theme(style="whitegrid", context="notebook")

SEED = 17
rng = np.random.default_rng(SEED)
algorithm_globals.random_seed = SEED

## The common classification task

We use the Iris dataset downloaded from Kaggle (you can also use the scikit-learn version of the dataset). To keep the effect of quantum encoding directly observable, we restrict the problem to
- `Iris-versicolor`;
- `Iris-virginica`.

We use two input features:
- petal length;
- petal width.

We deliberately exclude `Iris-setosa` because it is nearly trivially separable from the other species using petal measurements. The Versicolor--Virginica problem has a visible but imperfect class boundary, making information loss, periodicity, kernel geometry, and variational optimization easier to compare. The physical measurements will first be mapped into the unit square. Every encoding then receives the same scaled input

$$
u=(u_{\mathrm{length}},u_{\mathrm{width}})\in[0,1]^2.
$$

In [ ]:
# Setup the dataset
dataset_directory = Path(kagglehub.dataset_download("uciml/iris"))
csv_files = sorted(dataset_directory.glob("*.csv"))
dataset_path = csv_files[0]
iris = pd.read_csv(dataset_path)

# Display the dataset
display(iris.head())
display(iris.columns.to_series(name="column"))

In [ ]:
# We will only look at these columns
required_columns = ["PetalLengthCm", "PetalWidthCm", "Species"]

# Now set the labels what is 0 and what is 1
species_to_label = {
    "Iris-versicolor": 0,
    "Iris-virginica": 1,
}

# Locate the "Species" column and select only the versicolor and the virginica species in there
# drop everything else
data = iris.loc[iris["Species"].isin(species_to_label), required_columns].dropna()

# Now assign them the labels and drop everything else
data["label"] = data["Species"].map(species_to_label)
data = data.reset_index(drop=True)

# Display the new cleaned dataset
display(data.head())
display(data["Species"].value_counts())

Visualize how they look like in a scatterplot

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6), constrained_layout=True)

sns.scatterplot(
    data=data,
    x="PetalLengthCm",
    y="PetalWidthCm",
    hue="Species",
    palette={
        "Iris-versicolor": "#2A6FDB",
        "Iris-virginica": "#E4572E",
    },
    edgecolor="black",
    linewidth=0.5,
    s=70,
    ax=ax,
)

ax.set_xlabel("Petal length (cm)")
ax.set_ylabel("Petal width (cm)")
ax.set_title("Binary Iris classification task")

plt.show()

### Split into training and testing

There should be about 75 training observations after the split, so we do not need to reduce the dataset aggressively.

In [ ]:
# Features
X = data[["PetalLengthCm", "PetalWidthCm"]].to_numpy()
y = data["label"].to_numpy()

# The actual split
# The stratify=y thing specifies that data is split in a stratified fashion, using this as the class labels
X_train_raw, X_test_raw, y_train, y_test = train_test_split(X, y, test_size=0.25, stratify=y,random_state=SEED)

# Initialize a scalar to scale data and normalize its values
input_scaler = MinMaxScaler()
X_train_unit = input_scaler.fit_transform(X_train_raw)
X_test_unit = input_scaler.transform(X_test_raw)

print("Training shape:", X_train_unit.shape)
print("Test shape:", X_test_unit.shape)
print("Class counts in training set:", np.bincount(y_train))
print("Class counts in test set:", np.bincount(y_test))

In [ ]:
quantum_training_frame = pd.DataFrame(
    {
        "scaled_petal_length": X_train_unit[:, 0],
        "scaled_petal_width": X_train_unit[:, 1],
        "label": y_train,
    }
)

# Just making sure this is balanced
N_PER_CLASS = 35

grouped_data = quantum_training_frame.groupby("label", group_keys=False)
quantum_training_frame = grouped_data.sample(n=N_PER_CLASS, random_state=SEED)
quantum_training_frame = quantum_training_frame.sample(frac=1.0, random_state=SEED,)

quantum_training_frame = quantum_training_frame.reset_index(drop=True)

# Specify the dataset to quantum
X_train_quantum = quantum_training_frame[["scaled_petal_length", "scaled_petal_width"]].to_numpy()
y_train_quantum = quantum_training_frame["label"].to_numpy()

print("Quantum training shape:", X_train_quantum.shape)
display(quantum_training_frame["label"].value_counts())

## What exactly are we studying?

A quantum machine-learning model contains several conceptually different layers. In this notebook, we study three of them:

- the quantum states produced by an encoding;
- the similarity measure induced by those states;
- the classification functions produced after adding a trainable circuit.

These objects are related, but they are not interchangeable.

### The states reached by the encoding

We begin with a classical data point $x\in\mathcal{X}$. An encoding circuit $U_\phi(x)$ maps this point to a quantum state:

$$
|\phi(x)\rangle
=
U_\phi(x)|0\rangle^{\otimes n}.
$$

As $x$ varies over the input space, the circuit produces a collection of quantum states:

$$
\mathcal{M}_\phi
=
\left\{
|\phi(x)\rangle
:
x\in\mathcal{X}
\right\}.
$$

We call $\mathcal{M}_\phi$ the reachable state set of the encoding.

The full state space of $n$ qubits has dimension $2^n$, but this does not mean that the data can reach every state in that space. The data may trace out only a curve, a surface, a finite collection of basis states, or some other restricted subset.

This is the first question we study:

> As the classical data vary, which quantum states can the encoding actually reach?

Different encodings answer this question differently. Basis encoding reaches a discrete collection of computational-basis states. Angle encoding produces a continuous family of product states. Dense-angle encoding wraps a two-dimensional input space around a single Bloch sphere. An entangling feature map can reach states that cannot be written as products of individual qubit states.

### The similarity geometry created by the encoding

After encoding two inputs $x$ and $z$, we can compare their quantum states using their fidelity:

$$
K_\phi(x,z)
=
\left|
\langle\phi(x)|\phi(z)\rangle
\right|^2.
$$

This quantity is the fidelity quantum kernel.

It answers a different question:

> According to the encoding, how similar are two classical data points?

The encoding therefore imposes a new geometry on the dataset. Two points that are close in the original feature space need not remain close after encoding. Conversely, two distinct inputs may be mapped to the same quantum state and become indistinguishable.

To see why this is a kernel, define the density operator

$$
\rho_\phi(x)
=
|\phi(x)\rangle\langle\phi(x)|.
$$

The fidelity can then be written as

$$
K_\phi(x,z)
=
\operatorname{Tr}
\left[
\rho_\phi(x)\rho_\phi(z)
\right].
$$

Thus, the quantum kernel is an inner product between encoded density operators. The classifier does not operate directly on the original coordinates. It operates on the similarity geometry created by the encoding.

In this notebook, we inspect that geometry using kernel matrices, kernel eigenvalues, class alignment, effective rank, and distances between encoded states.

### The classification functions available to the model

Encoding the data does not by itself produce a classifier. We must still process the encoded state and measure an output.

For a variational quantum classifier, we apply a trainable circuit $V(\theta)$ after the encoding:

$$
|\psi(x,\theta)\rangle
=
V(\theta)U_\phi(x)|0\rangle^{\otimes n}.
$$

A measurement then produces a real-valued classification score $f_\theta(x)$. For a binary classifier, we may define

$$
f_\theta(x)
=
p_\theta(1\mid x)
-
p_\theta(0\mid x).
$$

The predicted class changes where

$$
f_\theta(x)=0.
$$

As the trainable parameters $\theta$ vary, the model can generate a family of decision functions:

$$
\mathcal{F}_{\phi,V}
=
\left\{
f_\theta(x)
:
\theta\in\Theta
\right\}.
$$

This is the reachable function family of the variational model.

It depends on both parts of the circuit:

$$
\text{reachable functions}
=
\text{encoding}
+
\text{trainable ansatz}
+
\text{measurement}.
$$

The encoding determines how the classical variables enter the circuit. The ansatz determines how those encoded states can subsequently be transformed. The measurement determines which information is converted into a class score.

This leads to the third question we study:

> Given an encoding and a fixed trainable circuit, which decision boundaries can the classifier represent?

### The distinction we will maintain

The logical flow of the notebook is therefore

$$
x
\longmapsto
|\phi(x)\rangle
\longmapsto
K_\phi(x,z)
\longmapsto
f(x).
$$

The reachable quantum states describe where the encoding sends the data.

The quantum kernel describes the similarity geometry among those states.

The decision function describes how a learning algorithm uses that geometry to separate the classes.

A feature map may reach many distinguishable states without arranging them according to the labels. A kernel may have a large effective rank without producing a useful separator. A useful family of decision functions may exist even though a variational optimizer fails to find one.

For this reason, we will examine state reachability, kernel geometry, separator expressivity, and empirical trainability separately.

## Encoding transformations used in this experiment

Let

$$
u
=
\left(
u_{\mathrm{length}},
u_{\mathrm{width}}
\right)
\in[0,1]^2
$$

denote the scaled petal length and petal width of an Iris flower.

The same scaled input $u$ is supplied to every encoding. Each encoding then transforms these two numbers into circuit parameters or quantum-state amplitudes in a different way.

| Encoding | Transformation of the Iris features | Qubits | Main information effect |
|---|---|---:|---|
| Basis | Place each petal feature into one of three bins and write the bin index using two bits | 4 | Replaces continuous measurements with discrete categories |
| Amplitude | Normalize the padded vector $(1,u_{\mathrm{length}},u_{\mathrm{width}},0)$ | 2 | Represents both petal measurements through relative amplitudes |
| Angle | $(\pi u_{\mathrm{length}},\pi u_{\mathrm{width}})$ | 2 | Maps each petal feature to a separate qubit rotation |
| Phase | $(2\pi u_{\mathrm{length}},2\pi u_{\mathrm{width}})$ | 2 | Maps each feature to a periodic phase |
| Dense angle | $(\pi u_{\mathrm{length}},2\pi u_{\mathrm{width}})$ | 1 | Places both petal features on one Bloch sphere |
| $ZZ$ | $(\pi u_{\mathrm{length}},\pi u_{\mathrm{width}})$ | 2 | Adds a joint, nonlinear dependence on petal length and width |



### Basis encoding

Each continuous petal measurement is first assigned to one of three bins:
$$
0,\qquad 1,\qquad 2.
$$

The bin index is then represented by two classical bits:
$$
0\longmapsto 00,
\qquad
1\longmapsto 01,
\qquad
2\longmapsto 10.
$$
Since we use two petal features, the complete encoded bit string contains four bits and therefore requires four qubits. The basis encoding deliberately discards information within each bin. Two flowers with different petal measurements prepare exactly the same quantum state whenever both measurements fall into the same pair of bins.

In [ ]:
def basis_coordinates(X:np.ndarray, bins: int, return_intermediates:bool=False):
    """Convert scaled Iris measurements into basis-encoding coordinates.

    Each flower is represented by its scaled petal length and petal width.
    Each feature is assigned to one of four bins, and each bin index is
    represented using two bits. The final encoding therefore contains four
    bits per flower.

    Args:
        X: Input data with shape ``(n_samples, 2)`` or ``(2,)``. Each row
            must contain scaled petal length followed by scaled petal width.
            All values must lie in the interval ``[0, 1]``.
        return_intermediates: Whether to return the intermediate binning and
            bit-extraction results in addition to the encoded coordinates.

    Returns:
        If ``return_intermediates`` is ``False``, returns a floating-point
        array with shape ``(n_samples, 4)`` containing

        ``[length high bit, length low bit, width high bit, width low bit]``.

        If ``return_intermediates`` is ``True``, returns a tuple containing
        the encoded array and a dictionary of intermediate arrays.

    Raises:
        ValueError: If the input does not contain exactly two features.
        ValueError: If any input value lies outside the interval ``[0, 1]``.
    """
    # Convert the input into a floating-point NumPy array.
    input_coordinates = np.asarray(X, dtype=float)

    # Convert one flower from shape (2,) into shape (1, 2).
    # This is done for demo purposes only. Ignore otherwise
    input_coordinates = np.atleast_2d(input_coordinates)

    # Map each feature from [0, 1] into [0, 4].
    scaled_bin_positions = bins * input_coordinates

    # Use the integer part to obtain the preliminary bin index.
    preliminary_bin_indices = np.floor(scaled_bin_positions).astype(int)

    # Map the endpoint x = 1 from index 4 back to the final valid bin.
    bin_indices = np.clip(preliminary_bin_indices, 0, bins - 1)

    # Separate the bin indices for the two Iris features.
    petal_length_bins = bin_indices[:, 0]
    petal_width_bins = bin_indices[:, 1]

    # Extract the two binary digits of the petal-length bin.
    petal_length_high_bit = (petal_length_bins >> 1) & 1
    petal_length_low_bit = petal_length_bins & 1

    # Extract the two binary digits of the petal-width bin.
    petal_width_high_bit = (petal_width_bins >> 1) & 1
    petal_width_low_bit = petal_width_bins & 1

    # Group the two bits associated with petal length.
    petal_length_bits = np.column_stack(
        [
            petal_length_high_bit,
            petal_length_low_bit,
        ]
    )

    # Group the two bits associated with petal width.
    petal_width_bits = np.column_stack(
        [
            petal_width_high_bit,
            petal_width_low_bit,
        ]
    )

    # Combine both two-bit encodings into one four-bit vector.
    encoded_bits = np.column_stack(
        [
            petal_length_bits,
            petal_width_bits,
        ]
    ).astype(float)

    # Return only the encoded array during ordinary preprocessing.
    if not return_intermediates:
        return encoded_bits

    # Collect the intermediate values for demonstrations and debugging.
    intermediates = {
        "input_coordinates": input_coordinates,
        "scaled_bin_positions": scaled_bin_positions,
        "preliminary_bin_indices": preliminary_bin_indices,
        "bin_indices": bin_indices,
        "petal_length_bins": petal_length_bins,
        "petal_width_bins": petal_width_bins,
        "petal_length_bits": petal_length_bits,
        "petal_width_bits": petal_width_bits,
    }

    return encoded_bits, intermediates

In [ ]:
N_BASIS_BINS = 4
example_flower = [0.40, 0.80]
encoded_bits, basis_steps = basis_coordinates(example_flower, bins=N_BASIS_BINS, return_intermediates=True)
print("Scaled Iris features:", example_flower)
print("Basis coordinates:", encoded_bits)

In [ ]:
print("Original scaled input")
print(basis_steps["input_coordinates"])

print("\nPositions after multiplying by four")
print(basis_steps["scaled_bin_positions"])

print("\nPreliminary bin indices obtained with floor")
print(basis_steps["preliminary_bin_indices"])

print("\nFinal bin indices after clipping")
print(basis_steps["bin_indices"])

print("\nPetal-length bits")
print(basis_steps["petal_length_bits"])

print("\nPetal-width bits")
print(basis_steps["petal_width_bits"])

print("\nComplete four-bit encoding")
print(encoded_bits)

## From an Iris flower to a computational-basis state

Consider a flower with scaled petal measurements

$$
u
=
\left(
u_{\mathrm{length}},
u_{\mathrm{width}}
\right)
=
(0.40,0.80).
$$

Each coordinate is placed into one of four bins using

$$
q(u)
=
\min
\left(
\left\lfloor4u\right\rfloor,
3
\right).
$$

For petal length,

$$
q(0.40)
=
\left\lfloor1.60\right\rfloor
=
1
=
01_2.
$$

For petal width,

$$
q(0.80)
=
\left\lfloor3.20\right\rfloor
=
3
=
11_2.
$$

Combining the two binary representations gives

$$
b(u)
=
(0,1,1,1).
$$

The encoded quantum state is therefore

$$
|\phi_{\mathrm{basis}}(u)\rangle
=
|0111\rangle.
$$

The important operation is quantization. Every flower in the same two-dimensional bin is mapped to the same computational-basis state. Basis encoding therefore preserves the identity of the bin but discards every distinction between flowers inside that bin.

In [ ]:
from matplotlib.patches import Rectangle

fig, axes = plt.subplots(1, 2, figsize=(20, 8), constrained_layout=True)

# LEFT PANEL
feature_values = np.linspace(0.0, 1.0, 1001)
one_dimensional_bins = np.floor(N_BASIS_BINS * feature_values).astype(int)
one_dimensional_bins = np.clip(one_dimensional_bins, 0, N_BASIS_BINS - 1)

axes[0].step(feature_values, one_dimensional_bins, where="post", linewidth=2.5, color="#2A6FDB")
for boundary in np.linspace(0, 1, N_BASIS_BINS + 1):
    axes[0].axvline(boundary, color="black", linestyle=":", alpha=0.5)
axes[0].set_xlim(0, 1)
axes[0].set_ylim(-0.4, 3.4)
axes[0].set_yticks(
    [0, 1, 2, 3],
    [
        "bin 0 = 00",
        "bin 1 = 01",
        "bin 2 = 10",
        "bin 3 = 11",
    ],
)
axes[0].set_xlabel("Scaled feature value")
axes[0].set_ylabel("Assigned bin")
axes[0].set_title("One continuous feature becomes two bits")

# Right panel: quantization of the complete Iris feature plane.
cell_width = 1.0 / N_BASIS_BINS
cell_colors = plt.get_cmap("tab20").colors

for length_bin in range(N_BASIS_BINS):
    for width_bin in range(N_BASIS_BINS):
        lower_length = length_bin * cell_width
        lower_width = width_bin * cell_width

        cell_number = (
            N_BASIS_BINS * length_bin
            + width_bin
        )

        rectangle = Rectangle(
            (lower_length, lower_width),
            cell_width,
            cell_width,
            facecolor=cell_colors[cell_number],
            edgecolor="black",
            linewidth=1.0,
            alpha=0.16,
        )

        axes[1].add_patch(rectangle)

        length_center = lower_length + cell_width / 2
        width_center = lower_width + cell_width / 2

        cell_bits = basis_coordinates(
            [length_center, width_center], bins=N_BASIS_BINS,
        )[0].astype(int)

        bit_string = "".join(
            cell_bits.astype(str)
        )

        axes[1].text(
            length_center,
            width_center,
            rf"$|{bit_string}\rangle$",
            ha="center",
            va="center",
            fontsize=10,
        )

sns.scatterplot(
    x=X_train_quantum[:, 0],
    y=X_train_quantum[:, 1],
    hue=y_train_quantum,
    palette={
        0: "#2A6FDB",
        1: "#E4572E",
    },
    edgecolor="black",
    linewidth=0.5,
    s=60,
    ax=axes[1],
)

axes[1].set_xlim(0.0, 1.0)
axes[1].set_ylim(0.0, 1.0)
axes[1].set_aspect("equal")

axes[1].set_xlabel("Scaled petal length")
axes[1].set_ylabel("Scaled petal width")
axes[1].set_title("The Iris feature plane becomes 16 basis states")

handles, _ = axes[1].get_legend_handles_labels()
axes[1].legend(
    handles=handles,
    labels=["Versicolor", "Virginica"],
    title="Species",
)

plt.show()


## Amplitude Encoding

The two scaled measurements are augmented and padded to form

$$
a(u)
=
\left(
1,
u_{\mathrm{length}},
u_{\mathrm{width}},
0
\right).
$$

The vector is normalized before it is loaded into the amplitudes:

$$
|\phi_{\mathrm{amp}}(u)\rangle
=
\frac{
|00\rangle
+
u_{\mathrm{length}}|01\rangle
+
u_{\mathrm{width}}|10\rangle
}{
\sqrt{
1+
u_{\mathrm{length}}^2+
u_{\mathrm{width}}^2
}
}.
$$

The constant component prevents normalization from erasing all information about the magnitude of the original two-dimensional vector. The final zero pads the vector to dimension four so that it can be represented by two qubits.


In [ ]:
def amplitude_coordinates(X, return_intermediates=False):
    """Construct normalized amplitude vectors from Iris measurements.

    The scaled petal measurements are embedded into the four-dimensional
    vector ``(1, petal length, petal width, 0)``. Each vector is then
    normalized to have unit Euclidean norm.

    Args:
        X: Input data with shape ``(n_samples, 2)`` or ``(2,)``. Each row
            contains scaled petal length followed by scaled petal width.
        return_intermediates: Whether to return the intermediate augmented
            vectors, norms, amplitudes, and probabilities.

    Returns:
        If ``return_intermediates`` is ``False``, returns the normalized
        amplitude vectors with shape ``(n_samples, 4)``.

        If ``return_intermediates`` is ``True``, returns a tuple containing
        the normalized amplitude vectors and a dictionary of intermediate
        arrays.

    Raises:
        ValueError: If the input does not contain exactly two features.
        ValueError: If any input value lies outside the interval ``[0, 1]``.
    """
    # Convert the input into a floating-point NumPy array.
    input_coordinates = np.asarray(X, dtype=float)

    # Convert one flower from shape (2,) into shape (1, 2) for the demo
    input_coordinates = np.atleast_2d(input_coordinates)


    # Add a constant coordinate and one padding coordinate.
    #
    # [petal length, petal width]
    # becomes
    # [1, petal length, petal width, 0].
    augmented_vectors = np.column_stack(
        [
            np.ones(len(input_coordinates)),
            input_coordinates[:, 0],
            input_coordinates[:, 1],
            np.zeros(len(input_coordinates)),
        ]
    )

    # Calculate the Euclidean norm of each augmented vector.
    vector_norms = np.linalg.norm(augmented_vectors, axis=1, keepdims=True)

    # Divide every vector by its norm so that its squared amplitudes sum to 1.
    normalized_amplitudes = augmented_vectors / vector_norms

    # Squaring the amplitudes gives computational-basis probabilities.
    basis_probabilities = np.abs(normalized_amplitudes) ** 2


    # Return only the amplitudes during normal preprocessing.
    if not return_intermediates:
        return normalized_amplitudes

    # Return the complete calculation for demonstrations.
    intermediates = {
        "input_coordinates": input_coordinates,
        "augmented_vectors": augmented_vectors,
        "vector_norms": vector_norms,
        "normalized_amplitudes": normalized_amplitudes,
        "basis_probabilities": basis_probabilities,
    }

    return normalized_amplitudes, intermediates

In [ ]:
# Demoing this
example_flower = [0.40, 0.80]

encoded_amplitudes, amplitude_steps = amplitude_coordinates(example_flower, return_intermediates=True)
basis_states = [
    "|00>",
    "|01>",
    "|10>",
    "|11>",
]

print("Original scaled Iris measurements:")
print(amplitude_steps["input_coordinates"])
print("\nAugmented vector:")
print(amplitude_steps["augmented_vectors"])

print("\nEuclidean norm:")
print(amplitude_steps["vector_norms"])

print("\nNormalized amplitude vector:")
print(encoded_amplitudes)

print("\nSum of measurement probabilities:")
print(amplitude_steps["basis_probabilities"].sum(axis=1))



## How does the amplitude-coordinate transformation work?

Consider a flower with scaled petal measurements

$$
u
=
\left(
u_{\mathrm{length}},
u_{\mathrm{width}}
\right)
=
(0.40,0.80).
$$

### Constructing a four-dimensional vector

A two-qubit quantum state has four computational-basis states:

$$
|00\rangle,
\qquad
|01\rangle,
\qquad
|10\rangle,
\qquad
|11\rangle.
$$

It therefore has four amplitudes. We construct the classical vector

$$
a(u)
=
\left(
1,
u_{\mathrm{length}},
u_{\mathrm{width}},
0
\right).
$$

For this flower,

$$
a(u)
=
(1,0.40,0.80,0).
$$

The first coordinate is a constant bias coordinate. The final coordinate is padding that increases the vector dimension to four.

### Why normalization is required

A valid quantum state must satisfy

$$
\langle\psi|\psi\rangle=1.
$$

Equivalently, its amplitudes must obey

$$
\sum_j |\alpha_j|^2=1.
$$

The unnormalized Iris vector does not yet satisfy this condition. Its Euclidean norm is

$$
\|a(u)\|
=
\sqrt{
1^2
+
0.40^2
+
0.80^2
+
0^2
}.
$$

Therefore,

$$
\|a(u)\|
=
\sqrt{1.80}
\approx
1.3416.
$$

We divide every coordinate by this norm:

$$
\widetilde{a}(u)
=
\frac{a(u)}{\|a(u)\|}.
$$

Numerically,

$$
\widetilde{a}(u)
\approx
(0.7454,0.2981,0.5963,0).
$$

### Constructing the quantum state

The four normalized coordinates become the four computational-basis amplitudes:

$$
|\phi_{\mathrm{amp}}(u)\rangle
=
0.7454|00\rangle
+
0.2981|01\rangle
+
0.5963|10\rangle
+
0|11\rangle.
$$

More generally,

$$
|\phi_{\mathrm{amp}}(u)\rangle
=
\frac{
|00\rangle
+
u_{\mathrm{length}}|01\rangle
+
u_{\mathrm{width}}|10\rangle
}{
\sqrt{
1+
u_{\mathrm{length}}^2+
u_{\mathrm{width}}^2
}
}.
$$

The quantum state contains the petal measurements in its relative amplitudes.

### Measurement probabilities

Measuring the state in the computational basis does not return the amplitudes directly. It produces probabilities obtained by squaring their magnitudes:

$$
p_j
=
|\alpha_j|^2.
$$

For this example,

$$
p(00)
=
|0.7454|^2
\approx
0.5556,
$$

$$
p(01)
=
|0.2981|^2
\approx
0.0889,
$$

$$
p(10)
=
|0.5963|^2
\approx
0.3556,
$$

and

$$
p(11)=0.
$$

These probabilities satisfy

$$
p(00)+p(01)+p(10)+p(11)=1.
$$

Amplitude encoding does not mean that one measurement reveals the complete classical feature vector. The encoded information is distributed across the quantum amplitudes. Accessing that information requires repeated measurements or interference with another quantum state.

In [ ]:
# This is all operating on the intermediates dict
augmented_vector = amplitude_steps["augmented_vectors"][0]
normalized_vector = amplitude_steps["normalized_amplitudes"][0]
measurement_probabilities = amplitude_steps["basis_probabilities"][0]

# Create basis labels for viz
basis_labels = [
    r"$|00\rangle$",
    r"$|01\rangle$",
    r"$|10\rangle$",
    r"$|11\rangle$",
]

fig, axes = plt.subplots(1, 3, figsize=(20, 5.4), constrained_layout=True)
# Unnormalized classical coordinates.
axes[0].bar(basis_labels, augmented_vector, color="#7AA6DC", edgecolor="black")
axes[0].set_ylim(0.0, 1.1)
axes[0].set_ylabel("Coordinate value")
axes[0].set_title("Augmented classical vector")

for index, value in enumerate(augmented_vector):
    axes[0].text(index, value + 0.035, f"{value:.2f}", ha="center")

# Normalized quantum amplitudes.
axes[1].bar(basis_labels, normalized_vector, color="#6CCB9F", edgecolor="black")
axes[1].set_ylim(0.0, 1.1)
axes[1].set_ylabel("Quantum amplitude")
axes[1].set_title("Normalized state amplitudes")

for index, value in enumerate(normalized_vector):
    axes[1].text(index, value + 0.035, f"{value:.3f}", ha="center")

# Computational-basis measurement probabilities.
axes[2].bar(basis_labels, measurement_probabilities, color="#F29E67", edgecolor="black")
axes[2].set_ylim(0.0, 1.1)
axes[2].set_ylabel("Measurement probability")
axes[2].set_title("Squared amplitudes")

for index, value in enumerate(measurement_probabilities):
    axes[2].text(index, value + 0.035, f"{value:.3f}", ha="center")

fig.suptitle(
    "Amplitude encoding of the scaled Iris point (0.40, 0.80)",
    fontsize=15,
)

plt.show()


## Preparing the amplitude-encoded state

The previous visualization calculated the desired amplitudes but did not construct a quantum circuit.

We now perform two demonstrations.

First, we construct a general amplitude-encoding routine that can encode an arbitrary real or complex vector. If the vector dimension is not a power of two, zeros are appended until it can be represented by an integer number of qubits.

Second, we apply the same routine to an actual flower from the Iris dataset. We decompose its state-preparation circuit into elementary gates and inspect the quantum state after each gate.

A general two-qubit state cannot be represented completely by one Bloch sphere. The Bloch sphere represents one qubit. For the two-qubit state, we therefore display:

- one Bloch sphere for each qubit's reduced state;
- a Q-sphere for the complete joint state.

In [ ]:
import ipywidgets as widgets

from IPython.display import display
from qiskit import QuantumCircuit, transpile
from qiskit.circuit.library import StatePreparation
from qiskit.quantum_info import (
    DensityMatrix,
    Statevector,
    concurrence,
    partial_trace,
    purity,
)
from qiskit.visualization import (
    plot_bloch_multivector,
    plot_state_qsphere,
)

In [ ]:
def build_amplitude_encoding(data):
    """Construct a quantum circuit that amplitude-encodes a data vector.

    The input vector is padded with zeros until its dimension is a power
    of two. It is then normalized and supplied to a Qiskit
    ``StatePreparation`` instruction.

    Args:
        data: One-dimensional real or complex data vector.

    Returns:
        A tuple containing:

        - the amplitude-encoding circuit;
        - the resulting statevector;
        - a dictionary containing the intermediate calculations.

    Raises:
        ValueError: If the input is not one-dimensional.
        ValueError: If the input vector is empty.
        ValueError: If every element of the input vector is zero.
    """
    # Convert the input into a one-dimensional complex NumPy array.
    input_vector = np.asarray(data,dtype=complex)


    # Determine the smallest number of qubits whose Hilbert space
    # can contain every coordinate of the input vector.
    number_of_qubits = max(1, int(np.ceil(np.log2(len(input_vector)))))

    hilbert_space_dimension = 2 ** number_of_qubits

    # Determine how many zeros must be appended.
    number_of_padding_zeros = hilbert_space_dimension - len(input_vector)

    # Pad the vector to a power-of-two dimension.
    padded_vector = np.pad(input_vector, pad_width=(0, number_of_padding_zeros), mode="constant", constant_values=0.0)

    # Calculate the Euclidean norm required for state normalization.
    vector_norm = np.linalg.norm(padded_vector)

    if np.isclose(vector_norm, 0.0):
        raise ValueError(
            "The zero vector cannot be amplitude encoded."
        )

    # Normalize the vector so that the squared amplitudes sum to one.
    normalized_vector = padded_vector / vector_norm

    # Construct a state-preparation circuit.
    preparation = StatePreparation(normalized_vector)

    circuit = QuantumCircuit(number_of_qubits, name="AmplitudeEncoding")

    circuit.append(preparation, range(number_of_qubits))

    # Obtain the exact state produced by the circuit.
    statevector = Statevector.from_instruction(circuit)

    intermediates = {
        "input_vector": input_vector,
        "number_of_qubits": number_of_qubits,
        "hilbert_space_dimension": hilbert_space_dimension,
        "number_of_padding_zeros": number_of_padding_zeros,
        "padded_vector": padded_vector,
        "vector_norm": vector_norm,
        "normalized_vector": normalized_vector,
        "basis_probabilities": (
            np.abs(normalized_vector) ** 2
        ),
    }

    return circuit, statevector, intermediates

In [ ]:
general_data_vector = np.array([4.0, 8.0, 5.0])

general_amplitude_circuit, general_amplitude_state, general_amplitude_steps = build_amplitude_encoding(general_data_vector)

print("Input vector:", np.real_if_close(general_amplitude_steps["input_vector"]))
print("Number of qubits:", general_amplitude_steps["number_of_qubits"])
print("Hilbert-space dimension:", general_amplitude_steps["hilbert_space_dimension"])
print("Number of padding zeros:", general_amplitude_steps["number_of_padding_zeros"])
print("Padded vector:", np.real_if_close(general_amplitude_steps["padded_vector"]))
print("Vector norm:", general_amplitude_steps["vector_norm"])
print("Normalized vector:", np.real_if_close(general_amplitude_steps["normalized_vector"]))
print("Measurement probabilities:", general_amplitude_steps["basis_probabilities"])
print("Sum of probabilities:", general_amplitude_steps["basis_probabilities"].sum())

display(general_amplitude_circuit.decompose(reps=5).draw(output="mpl", fold=-1))

In [ ]:
compiled_amplitude_circuit = transpile(general_amplitude_circuit, basis_gates=["u", "cx"], optimization_level=0, seed_transpiler=SEED)

encoding_states = [Statevector.from_label("00")]
encoding_step_labels = [r"Initial state $|00\rangle$"]
current_state = encoding_states[0]

for step_number, circuit_instruction in enumerate(compiled_amplitude_circuit.data, start=1):
    # Extract the gate from Qiskit's circuit-instruction object.
    operation = circuit_instruction.operation

    # Convert the qubits acted upon by the gate into integer indices.
    qubit_indices = [compiled_amplitude_circuit.find_bit(qubit).index for qubit in circuit_instruction.qubits]

    # Apply the current gate to the previous statevector.
    current_state = current_state.evolve(operation, qargs=qubit_indices)

    # Save the new state so that it can be displayed as an animation frame.
    encoding_states.append(current_state)

    # Create a readable list of the qubits acted upon by the current gate.
    qubit_text = ", ".join(f"q{index}" for index in qubit_indices)

    # Save a label describing the gate associated with this animation frame.
    encoding_step_labels.append(f"Step {step_number}: {operation.name} on {qubit_text}")

final_state_fidelity = np.abs(np.vdot(general_amplitude_state.data, encoding_states[-1].data)) ** 2

display(compiled_amplitude_circuit.draw(output="mpl", fold=-1))
print("Number of elementary gates:", len(compiled_amplitude_circuit.data))
print("Fidelity between the final snapshot and the target state:", final_state_fidelity)

In [ ]:
def show_amplitude_encoding_step(step):
    """Display one state from the gate-by-gate encoding sequence.

    Args:
        step: Index of the circuit snapshot to display.
    """
    state = encoding_states[step]

    print(encoding_step_labels[step])
    print("Statevector:", np.round(state.data, decimals=4))
    print("\nThe Bloch spheres show the reduced states of the individual qubits.")

    bloch_figure = plot_bloch_multivector(state)
    display(bloch_figure)
    plt.close(bloch_figure)

    print("\nThe Q-sphere shows the complete two-qubit pure state.")

    qsphere_figure = plot_state_qsphere(state)
    display(qsphere_figure)
    plt.close(qsphere_figure)


play_button = widgets.Play(value=0, min=0, max=len(encoding_states) - 1, step=1, interval=1000, description="Play")
step_slider = widgets.IntSlider(value=0, min=0, max=len(encoding_states) - 1, step=1, description="Gate step", continuous_update=False)

widgets.jslink((play_button, "value"), (step_slider, "value"))

animation_output = widgets.interactive_output(show_amplitude_encoding_step, {"step": step_slider})

display(widgets.HBox([play_button, step_slider]), animation_output)

## Interpreting the moving demonstration

The animation follows the elementary gates obtained by decomposing the state-preparation circuit. Check that we have a 2 qubit state (because of padding and what not), so we this is the analysis on the reduced qubit state formed as a result of partially tracing out the other qubit.

The system begins in

$$
|\psi_0\rangle=|00\rangle.
$$

After applying the gate $G_k$, the state becomes

$$
|\psi_{k+1}\rangle=G_k|\psi_k\rangle.
$$

The final state is the amplitude encoding of the selected Iris flower:

$$
|\psi_{\mathrm{final}}\rangle=|\phi_{\mathrm{amp}}(u)\rangle.
$$

Single-qubit gates rotate the state of an individual qubit. Controlled gates can create correlations and entanglement between the two qubits.

The two Bloch spheres show the reduced states of the individual qubits. They do not show the complete two-qubit state. If a Bloch vector moves into the interior of its sphere, the corresponding reduced state is mixed. Because the complete state remains pure, this loss of local purity indicates entanglement with the other qubit.

The Q-sphere shows the amplitudes and relative phases of the complete two-qubit state. It therefore contains information that is absent from the two individual Bloch vectors.

# Angle encoding

Angle encoding represents a classical feature using the rotation angle of a qubit.

For a scaled feature $u\in[0,1]$, we define

$$
\theta(u)=\pi u.
$$

The encoded state is prepared by applying a $Y$-axis rotation:

$$
|\phi_{\mathrm{angle}}(u)\rangle=R_Y(\theta(u))|0\rangle.
$$

Because

$$
R_Y(\theta)|0\rangle=\cos\left(\frac{\theta}{2}\right)|0\rangle+\sin\left(\frac{\theta}{2}\right)|1\rangle,
$$

the feature is encoded as

$$
|\phi_{\mathrm{angle}}(u)\rangle=\cos\left(\frac{\pi u}{2}\right)|0\rangle+\sin\left(\frac{\pi u}{2}\right)|1\rangle.
$$

Unlike basis encoding, angle encoding does not divide the input into discrete bins. The encoded state changes continuously as the input feature changes.

In [ ]:
def angle_coordinates(X, return_intermediates=False):
    """Convert scaled Iris measurements into angle-encoding coordinates.

    Each scaled feature is multiplied by pi and used as the angle of a
    single-qubit Y rotation.

    Args:
        X: Input data with shape ``(n_samples, 2)`` or ``(2,)``. Each row
            contains scaled petal length followed by scaled petal width.
        return_intermediates: Whether to return the rotation angles,
            single-qubit amplitudes, and measurement probabilities.

    Returns:
        If ``return_intermediates`` is ``False``, returns the rotation
        angles with shape ``(n_samples, 2)``.

        If ``return_intermediates`` is ``True``, returns a tuple containing
        the rotation angles and a dictionary of intermediate arrays.

    Raises:
        ValueError: If the input does not contain exactly two features.
        ValueError: If any input value lies outside the interval ``[0, 1]``.
    """
    # Convert the input into a floating-point NumPy array.
    input_coordinates = np.asarray(X, dtype=float)

    # Convert one flower from shape (2,) into shape (1, 2).
    input_coordinates = np.atleast_2d(input_coordinates)

    # Verify that each flower has petal length and petal width.
    if input_coordinates.shape[1] != 2:
        raise ValueError("Expected two features: [scaled petal length, scaled petal width].")

    # Verify that the features have been scaled into the unit interval.
    if np.any(input_coordinates < 0.0) or np.any(input_coordinates > 1.0):
        raise ValueError("All input features must lie in the interval [0, 1].")

    # Convert each scaled feature into a rotation angle in [0, pi].
    rotation_angles = np.pi * input_coordinates

    # Calculate the |0> amplitude of each independently encoded qubit.
    zero_amplitudes = np.cos(rotation_angles / 2.0)

    # Calculate the |1> amplitude of each independently encoded qubit.
    one_amplitudes = np.sin(rotation_angles / 2.0)

    # Square the amplitudes to obtain single-qubit measurement probabilities.
    zero_probabilities = zero_amplitudes**2
    one_probabilities = one_amplitudes**2

    # Return only the angles during ordinary preprocessing.
    if not return_intermediates:
        return rotation_angles

    # Return the complete calculation for demonstrations.
    intermediates = {
        "input_coordinates": input_coordinates,
        "rotation_angles": rotation_angles,
        "zero_amplitudes": zero_amplitudes,
        "one_amplitudes": one_amplitudes,
        "zero_probabilities": zero_probabilities,
        "one_probabilities": one_probabilities,
    }

    return rotation_angles, intermediates

In [ ]:
example_flower = [0.40, 0.80]

example_angles, angle_steps = angle_coordinates(example_flower, return_intermediates=True)

angle_trace = pd.DataFrame({
    "Feature": ["Petal length", "Petal width"],
    "Scaled value": angle_steps["input_coordinates"][0],
    "Angle in radians": angle_steps["rotation_angles"][0],
    "Angle divided by pi": angle_steps["rotation_angles"][0] / np.pi,
    "Amplitude of |0>": angle_steps["zero_amplitudes"][0],
    "Amplitude of |1>": angle_steps["one_amplitudes"][0],
    "Probability of 0": angle_steps["zero_probabilities"][0],
    "Probability of 1": angle_steps["one_probabilities"][0],
})

display(angle_trace.style.format({
    "Scaled value": "{:.3f}",
    "Angle in radians": "{:.3f}",
    "Angle divided by pi": "{:.3f}",
    "Amplitude of |0>": "{:.3f}",
    "Amplitude of |1>": "{:.3f}",
    "Probability of 0": "{:.3f}",
    "Probability of 1": "{:.3f}",
}))

## Example: encoding the scaled flower $(0.40,0.80)$

For the scaled petal length,

$$
u_{\mathrm{length}}=0.40,
$$

so the rotation angle is

$$
\theta_{\mathrm{length}}=\pi(0.40)=0.40\pi.
$$

The petal-length qubit becomes

$$
|\phi_{\mathrm{length}}\rangle=\cos(0.20\pi)|0\rangle+\sin(0.20\pi)|1\rangle.
$$

For the scaled petal width,

$$
u_{\mathrm{width}}=0.80,
$$

so the rotation angle is

$$
\theta_{\mathrm{width}}=\pi(0.80)=0.80\pi.
$$

The petal-width qubit becomes

$$
|\phi_{\mathrm{width}}\rangle=\cos(0.40\pi)|0\rangle+\sin(0.40\pi)|1\rangle.
$$

Each feature controls the position of one qubit on a Bloch sphere.

## A single angle-encoded feature

The Bloch vector of the state

$$
R_Y(\theta)|0\rangle
$$

is

$$
r(\theta)=(\sin\theta,0,\cos\theta).
$$

After setting $\theta=\pi u$, the Bloch vector becomes

$$
r(u)=(\sin(\pi u),0,\cos(\pi u)).
$$

As $u$ increases from $0$ to $1$, the state moves along the $x$-$z$ meridian:

$$
u=0\quad\longmapsto\quad|0\rangle,
$$

$$
u=\frac{1}{2}\quad\longmapsto\quad|+\rangle,
$$

$$
u=1\quad\longmapsto\quad|1\rangle.
$$

The following demonstration allows the encoded feature to move continuously between these states.

In [ ]:
GENERAL_ANGLE_STEPS = 41

general_scaled_values = np.linspace(0.0, 1.0, GENERAL_ANGLE_STEPS)
general_rotation_angles = np.pi * general_scaled_values
general_angle_states = []

for rotation_angle in general_rotation_angles:
    circuit = QuantumCircuit(1)
    circuit.ry(rotation_angle, 0)
    general_angle_states.append(Statevector.from_instruction(circuit))

In [ ]:
def show_general_angle_step(step):
    """Display one state from the general angle-encoding trajectory.

    Args:
        step: Index of the state to display.
    """
    scaled_value = general_scaled_values[step]
    rotation_angle = general_rotation_angles[step]
    state = general_angle_states[step]

    print(f"Scaled feature: u = {scaled_value:.3f}")
    print(f"Rotation angle: theta = {rotation_angle:.3f} radians = {rotation_angle / np.pi:.3f} pi")
    print("Statevector:", np.round(state.data, decimals=4))

    bloch_figure = plot_bloch_multivector(state)
    display(bloch_figure)
    plt.close(bloch_figure)


general_angle_play = widgets.Play(value=0, min=0, max=GENERAL_ANGLE_STEPS - 1, step=1, interval=100, description="Play")
general_angle_slider = widgets.IntSlider(value=0, min=0, max=GENERAL_ANGLE_STEPS - 1, step=1, description="Position", continuous_update=False)

widgets.jslink((general_angle_play, "value"), (general_angle_slider, "value"))

general_angle_output = widgets.interactive_output(show_general_angle_step, {"step": general_angle_slider})

display(widgets.HBox([general_angle_play, general_angle_slider]), general_angle_output)

In [ ]:
scaled_values = np.linspace(0.0, 1.0, 500)
probability_zero = np.cos(np.pi * scaled_values / 2.0) ** 2
probability_one = np.sin(np.pi * scaled_values / 2.0) ** 2

fig, ax = plt.subplots(figsize=(9, 5), constrained_layout=True)

ax.plot(scaled_values, probability_zero, label=r"$p(0\mid u)$", linewidth=2.5, color="#2A6FDB")
ax.plot(scaled_values, probability_one, label=r"$p(1\mid u)$", linewidth=2.5, color="#E4572E")
ax.axvline(0.5, color="black", linestyle=":", alpha=0.5)

ax.set_xlabel("Scaled feature value")
ax.set_ylabel("Measurement probability")
ax.set_title("Measurement probabilities produced by angle encoding")
ax.set_ylim(-0.02, 1.02)
ax.legend()

plt.show()

## Measurement probabilities

The probability of measuring $0$ is

$$
p(0\mid u)=\cos^2\left(\frac{\pi u}{2}\right),
$$

while the probability of measuring $1$ is

$$
p(1\mid u)=\sin^2\left(\frac{\pi u}{2}\right).
$$

These are nonlinear but monotonic functions of $u$ over the chosen interval $[0,1]$.

At $u=0$, the state is $|0\rangle$. At $u=1$, the state is $|1\rangle$. The midpoint $u=1/2$ produces an equal superposition and therefore equal measurement probabilities.

## Angle encoding an actual Iris flower

We assign one qubit to each scaled Iris feature:

| Qubit | Feature | Rotation |
|---|---|---|
| $q_0$ | Petal length | $R_Y(\pi u_{\mathrm{length}})$ |
| $q_1$ | Petal width | $R_Y(\pi u_{\mathrm{width}})$ |

The feature-map circuit is

$$
U_{\mathrm{angle}}(u)=R_Y^{(0)}(\pi u_{\mathrm{length}})R_Y^{(1)}(\pi u_{\mathrm{width}}).
$$

Because the rotations act independently on different qubits, the encoded state is separable.

Using Qiskit's displayed bit order $|q_1q_0\rangle$, the state is

$$
|\phi_{\mathrm{angle}}(u)\rangle=|\phi_{\mathrm{width}}\rangle_{q_1}\otimes|\phi_{\mathrm{length}}\rangle_{q_0}.
$$

Angle encoding does not produce entanglement by itself.

In [ ]:
ANGLE_SAMPLE_INDEX = 0

species_names = {0: "Versicolor", 1: "Virginica"}

angle_raw_flower = X_train_raw[ANGLE_SAMPLE_INDEX]
angle_scaled_flower = X_train_unit[ANGLE_SAMPLE_INDEX]
angle_species = species_names[y_train[ANGLE_SAMPLE_INDEX]]

angle_flower_table = pd.DataFrame({
    "Feature": ["Petal length", "Petal width"],
    "Original measurement in cm": angle_raw_flower,
    "Scaled measurement": angle_scaled_flower,
})

print("Species:", angle_species)
display(angle_flower_table)

In [ ]:
actual_angles, actual_angle_steps = angle_coordinates(angle_scaled_flower, return_intermediates=True)

actual_angle_table = pd.DataFrame({
    "Feature": ["Petal length", "Petal width"],
    "Scaled value": actual_angle_steps["input_coordinates"][0],
    "Rotation angle": actual_angle_steps["rotation_angles"][0],
    "Rotation angle divided by pi": actual_angle_steps["rotation_angles"][0] / np.pi,
    "Amplitude of |0>": actual_angle_steps["zero_amplitudes"][0],
    "Amplitude of |1>": actual_angle_steps["one_amplitudes"][0],
})

display(actual_angle_table.style.format({
    "Scaled value": "{:.4f}",
    "Rotation angle": "{:.4f}",
    "Rotation angle divided by pi": "{:.4f}",
    "Amplitude of |0>": "{:.4f}",
    "Amplitude of |1>": "{:.4f}",
}))

In [ ]:
actual_angle_circuit = QuantumCircuit(2, name="IrisAngleEncoding")

actual_angle_circuit.ry(actual_angles[0, 0], 0)
actual_angle_circuit.ry(actual_angles[0, 1], 1)

actual_angle_state = Statevector.from_instruction(actual_angle_circuit)

display(actual_angle_circuit.draw(output="mpl", fold=-1))

print("Encoded statevector:", np.round(actual_angle_state.data, decimals=4))

In [ ]:
actual_angle_basis_states = ["|00>", "|01>", "|10>", "|11>"]
actual_angle_amplitudes = actual_angle_state.data
actual_angle_probabilities = np.abs(actual_angle_amplitudes) ** 2

actual_angle_state_table = pd.DataFrame({
    "Basis state": actual_angle_basis_states,
    "Amplitude": np.real_if_close(actual_angle_amplitudes),
    "Measurement probability": actual_angle_probabilities,
})

display(actual_angle_state_table.style.format({
    "Amplitude": "{:.4f}",
    "Measurement probability": "{:.4f}",
}))

print("Sum of probabilities:", actual_angle_probabilities.sum())

In [ ]:
actual_angle_bloch_figure = plot_bloch_multivector(actual_angle_state)
display(actual_angle_bloch_figure)
plt.close(actual_angle_bloch_figure)

actual_angle_qsphere_figure = plot_state_qsphere(actual_angle_state)
display(actual_angle_qsphere_figure)
plt.close(actual_angle_qsphere_figure)

In [ ]:
actual_angle_density_matrix = DensityMatrix(actual_angle_state)

angle_reduced_state_qubit_0 = partial_trace(actual_angle_density_matrix, [1])
angle_reduced_state_qubit_1 = partial_trace(actual_angle_density_matrix, [0])

angle_state_diagnostics = pd.Series({
    "Global-state purity": float(np.real(purity(actual_angle_density_matrix))),
    "Qubit-0 purity": float(np.real(purity(angle_reduced_state_qubit_0))),
    "Qubit-1 purity": float(np.real(purity(angle_reduced_state_qubit_1))),
    "Two-qubit concurrence": float(concurrence(actual_angle_state)),
}, name="Value")

display(angle_state_diagnostics.to_frame().style.format(precision=6))

## Why does angle encoding remain separable?

The feature-map state has the form

$$
|\phi_{\mathrm{angle}}(u)\rangle=|\phi_{\mathrm{width}}\rangle_{q_1}\otimes|\phi_{\mathrm{length}}\rangle_{q_0}.
$$

It is explicitly a tensor product of two single-qubit states. Its concurrence is therefore

$$
C\left(|\phi_{\mathrm{angle}}(u)\rangle\right)=0.
$$

The global state and both reduced states are pure:

$$
\operatorname{Tr}(\rho^2)=1,
$$

$$
\operatorname{Tr}(\rho_0^2)=1,
$$

$$
\operatorname{Tr}(\rho_1^2)=1.
$$

Both Bloch vectors lie on the surfaces of their spheres. This differs from the amplitude encoding used previously, which generally produced an entangled two-qubit state and mixed one-qubit reductions.

## Moving from $|00\rangle$ to the encoded Iris state

Let the final rotation angles be $\theta_{\mathrm{length}}$ and $\theta_{\mathrm{width}}$. We introduce a progress parameter $t\in[0,1]$ and define

$$
|\psi(t)\rangle=R_Y^{(0)}(t\theta_{\mathrm{length}})R_Y^{(1)}(t\theta_{\mathrm{width}})|00\rangle.
$$

At $t=0$,

$$
|\psi(0)\rangle=|00\rangle.
$$

At $t=1$,

$$
|\psi(1)\rangle=|\phi_{\mathrm{angle}}(u)\rangle.
$$

This is the continuous path generated by gradually increasing both rotation angles from zero to their final data-dependent values.

In [ ]:
ANGLE_ANIMATION_STEPS = 41

angle_progress_values = np.linspace(0.0, 1.0, ANGLE_ANIMATION_STEPS)
actual_angle_states = []

for progress in angle_progress_values:
    intermediate_circuit = QuantumCircuit(2)
    intermediate_circuit.ry(progress * actual_angles[0, 0], 0)
    intermediate_circuit.ry(progress * actual_angles[0, 1], 1)
    actual_angle_states.append(Statevector.from_instruction(intermediate_circuit))

In [ ]:
def show_actual_angle_step(step):
    """Display one state from the Iris angle-encoding trajectory.

    Args:
        step: Index of the trajectory state to display.
    """
    progress = angle_progress_values[step]
    length_angle = progress * actual_angles[0, 0]
    width_angle = progress * actual_angles[0, 1]
    state = actual_angle_states[step]

    print(f"Encoding progress: t = {progress:.3f}")
    print(f"Petal-length angle: {length_angle:.3f} radians")
    print(f"Petal-width angle: {width_angle:.3f} radians")
    print("Statevector:", np.round(state.data, decimals=4))

    bloch_figure = plot_bloch_multivector(state)
    display(bloch_figure)
    plt.close(bloch_figure)


actual_angle_play = widgets.Play(value=0, min=0, max=ANGLE_ANIMATION_STEPS - 1, step=1, interval=100, description="Play")
actual_angle_slider = widgets.IntSlider(value=0, min=0, max=ANGLE_ANIMATION_STEPS - 1, step=1, description="Progress", continuous_update=False)

widgets.jslink((actual_angle_play, "value"), (actual_angle_slider, "value"))

actual_angle_output = widgets.interactive_output(show_actual_angle_step, {"step": actual_angle_slider})

display(widgets.HBox([actual_angle_play, actual_angle_slider]), actual_angle_output)

## Interpreting the moving angle encoding

The first Bloch sphere represents the qubit assigned to petal length. The second represents the qubit assigned to petal width.

Each qubit moves along the $x$-$z$ meridian because the circuit uses only $R_Y$ rotations. No $y$ component is introduced into either Bloch vector.

The two Bloch vectors generally travel through different angles because the scaled petal length and petal width are different.

Both vectors remain on the surfaces of their Bloch spheres throughout the animation. This confirms that the individual qubits remain in pure states and that the angle-encoding circuit does not entangle them.

## Which states can angle encoding reach?

A general two-qubit state has four complex amplitudes:

$$
|\psi\rangle=\alpha_{00}|00\rangle+\alpha_{01}|01\rangle+\alpha_{10}|10\rangle+\alpha_{11}|11\rangle.
$$

Angle encoding cannot reach every such state. It reaches only states of the form

$$
|\phi_{\mathrm{angle}}(u)\rangle=R_Y(\pi u_{\mathrm{length}})|0\rangle\otimes R_Y(\pi u_{\mathrm{width}})|0\rangle.
$$

The reachable states satisfy three restrictions:

- all amplitudes are real;
- every state is separable;
- the complete state is determined by only two parameters.

Each qubit is also restricted to one meridian rather than the entire Bloch sphere.

The reachable set is therefore a two-dimensional product-state surface inside the larger two-qubit state space.

## The kernel induced by angle encoding

Consider two scaled flowers

$$
u=(u_{\mathrm{length}},u_{\mathrm{width}})
$$

and

$$
v=(v_{\mathrm{length}},v_{\mathrm{width}}).
$$

For one qubit,

$$
\langle\phi(u_j)|\phi(v_j)\rangle=\cos\left(\frac{\pi(u_j-v_j)}{2}\right).
$$

The two-qubit overlap factorizes because the states are products:

$$
\langle\phi_{\mathrm{angle}}(u)|\phi_{\mathrm{angle}}(v)\rangle=\cos\left(\frac{\pi(u_{\mathrm{length}}-v_{\mathrm{length}})}{2}\right)\cos\left(\frac{\pi(u_{\mathrm{width}}-v_{\mathrm{width}})}{2}\right).
$$

The fidelity kernel is

$$
K_{\mathrm{angle}}(u,v)=\cos^2\left(\frac{\pi(u_{\mathrm{length}}-v_{\mathrm{length}})}{2}\right)\cos^2\left(\frac{\pi(u_{\mathrm{width}}-v_{\mathrm{width}})}{2}\right).
$$

The kernel varies continuously and treats the two feature directions independently.

In [ ]:
ANGLE_KERNEL_GRID_SIZE = 100

kernel_axis = np.linspace(0.0, 1.0, ANGLE_KERNEL_GRID_SIZE)
kernel_length, kernel_width = np.meshgrid(kernel_axis, kernel_axis)
angle_kernel_grid = np.column_stack([kernel_length.ravel(), kernel_width.ravel()])

angle_reference_flower = np.array([0.50, 0.50])
coordinate_differences = angle_kernel_grid - angle_reference_flower

angle_kernel_values = np.prod(np.cos(np.pi * coordinate_differences / 2.0) ** 2, axis=1)
angle_kernel_surface = angle_kernel_values.reshape(kernel_length.shape)

fig, ax = plt.subplots(figsize=(8, 6), constrained_layout=True)

contour = ax.contourf(kernel_length, kernel_width, angle_kernel_surface, levels=np.linspace(0.0, 1.0, 21), cmap="viridis")

ax.scatter(angle_reference_flower[0], angle_reference_flower[1], marker="*", s=220, color="white", edgecolor="black", label="Reference flower")
ax.set_xlabel("Scaled petal length")
ax.set_ylabel("Scaled petal width")
ax.set_title("Angle-kernel similarity to the reference point")
ax.legend()

fig.colorbar(contour, ax=ax, label=r"$K_{\mathrm{angle}}(u,u_{\mathrm{ref}})$")

plt.show()

In [ ]:
pairwise_differences = X_train_quantum[:, None, :] - X_train_quantum[None, :, :]
angle_kernel_matrix = np.prod(np.cos(np.pi * pairwise_differences / 2.0) ** 2, axis=2)

label_order = np.argsort(y_train_quantum)
ordered_angle_kernel = angle_kernel_matrix[np.ix_(label_order, label_order)]

fig, ax = plt.subplots(figsize=(7, 6), constrained_layout=True)

sns.heatmap(ordered_angle_kernel, vmin=0.0, vmax=1.0, cmap="mako", square=True, xticklabels=False, yticklabels=False, ax=ax)

ax.set_xlabel("Training flowers sorted by species")
ax.set_ylabel("Training flowers sorted by species")
ax.set_title("Angle-encoding fidelity kernel")

plt.show()

## What has angle encoding changed?

Basis encoding transformed the Iris plane into a finite grid of exactly distinguishable states. Angle encoding instead transforms the plane into a continuous family of overlapping product states.

Nearby flowers ordinarily have similar quantum states. Their fidelity decreases smoothly as their scaled petal measurements separate.

The kernel remains separable between petal length and petal width. Angle encoding does not directly create an interaction between the two features.

Nevertheless, a kernel classifier can construct nonlinear decision functions from weighted sums of these smooth similarity functions:

$$
f(u)=\sum_{i\in\mathrm{SV}}\alpha_i y_iK_{\mathrm{angle}}(u_i,u)+b.
$$

The encoding therefore has greater smooth geometric resolution than the basis map, but it remains restricted to a comparatively simple product-state feature space.

# Phase encoding

Phase encoding represents a classical feature using the relative phase between the $|0\rangle$ and $|1\rangle$ components of a qubit.

The phase gate is

$$
P(\phi)=\begin{pmatrix}1&0\\0&e^{i\phi}\end{pmatrix}.
$$

Applying this gate directly to $|0\rangle$ has no observable effect because

$$
P(\phi)|0\rangle=|0\rangle.
$$

We must first create a superposition using a Hadamard gate:

$$
H|0\rangle=|+\rangle=\frac{|0\rangle+|1\rangle}{\sqrt{2}}.
$$

The phase gate then produces

$$
P(\phi)|+\rangle=\frac{|0\rangle+e^{i\phi}|1\rangle}{\sqrt{2}}.
$$

For a scaled feature $u\in[0,1]$, we choose

$$
\phi(u)=2\pi u.
$$

The phase-encoded state is therefore

$$
|\phi_{\mathrm{phase}}(u)\rangle=\frac{|0\rangle+e^{i2\pi u}|1\rangle}{\sqrt{2}}.
$$

The feature changes the relative phase, not the magnitudes of the two amplitudes.

In [ ]:
def phase_coordinates(X, return_intermediates=False):
    """Convert scaled Iris measurements into phase-encoding coordinates.

    Each scaled feature is multiplied by ``2*pi`` and encoded as the
    relative phase of a single-qubit state.

    Args:
        X: Input data with shape ``(n_samples, 2)`` or ``(2,)``. Each row
            contains scaled petal length followed by scaled petal width.
        return_intermediates: Whether to return the phase angles,
            amplitudes, probabilities, and Bloch-vector coordinates.

    Returns:
        If ``return_intermediates`` is ``False``, returns the phase angles
        with shape ``(n_samples, 2)``.

        If ``return_intermediates`` is ``True``, returns a tuple containing
        the phase angles and a dictionary of intermediate arrays.

    Raises:
        ValueError: If the input does not contain exactly two features.
        ValueError: If any input value lies outside the interval ``[0, 1]``.
    """
    # Convert the input into a floating-point NumPy array.
    input_coordinates = np.asarray(X, dtype=float)

    # Convert one flower from shape (2,) into shape (1, 2).
    input_coordinates = np.atleast_2d(input_coordinates)

    # Verify that each flower has petal length and petal width.
    if input_coordinates.shape[1] != 2:
        raise ValueError("Expected two features: [scaled petal length, scaled petal width].")

    # Verify that the features have been scaled into the unit interval.
    if np.any(input_coordinates < 0.0) or np.any(input_coordinates > 1.0):
        raise ValueError("All input features must lie in the interval [0, 1].")

    # Convert each scaled feature into a phase in [0, 2*pi].
    phase_angles = 2.0 * np.pi * input_coordinates

    # The amplitude of |0> is always 1/sqrt(2).
    zero_amplitudes = np.full_like(phase_angles, 1.0 / np.sqrt(2.0), dtype=complex)

    # The amplitude of |1> carries the data-dependent complex phase.
    one_amplitudes = np.exp(1j * phase_angles) / np.sqrt(2.0)

    # Computational-basis probabilities remain equal to one half.
    zero_probabilities = np.abs(zero_amplitudes) ** 2
    one_probabilities = np.abs(one_amplitudes) ** 2

    # The encoded states lie on the equator of the Bloch sphere.
    bloch_x = np.cos(phase_angles)
    bloch_y = np.sin(phase_angles)
    bloch_z = np.zeros_like(phase_angles)

    # Return only the phase angles during ordinary preprocessing.
    if not return_intermediates:
        return phase_angles

    # Return the complete calculation for demonstrations.
    intermediates = {
        "input_coordinates": input_coordinates,
        "phase_angles": phase_angles,
        "zero_amplitudes": zero_amplitudes,
        "one_amplitudes": one_amplitudes,
        "zero_probabilities": zero_probabilities,
        "one_probabilities": one_probabilities,
        "bloch_x": bloch_x,
        "bloch_y": bloch_y,
        "bloch_z": bloch_z,
    }

    return phase_angles, intermediates

In [ ]:
example_flower = [0.40, 0.80]

example_phases, phase_steps = phase_coordinates(example_flower, return_intermediates=True)

phase_trace = pd.DataFrame({
    "Feature": ["Petal length", "Petal width"],
    "Scaled value": phase_steps["input_coordinates"][0],
    "Phase in radians": phase_steps["phase_angles"][0],
    "Phase divided by pi": phase_steps["phase_angles"][0] / np.pi,
    "Real part of |1> amplitude": phase_steps["one_amplitudes"][0].real,
    "Imaginary part of |1> amplitude": phase_steps["one_amplitudes"][0].imag,
    "Probability of 0": phase_steps["zero_probabilities"][0],
    "Probability of 1": phase_steps["one_probabilities"][0],
    "Bloch x": phase_steps["bloch_x"][0],
    "Bloch y": phase_steps["bloch_y"][0],
})

display(phase_trace.style.format({
    "Scaled value": "{:.3f}",
    "Phase in radians": "{:.3f}",
    "Phase divided by pi": "{:.3f}",
    "Real part of |1> amplitude": "{:.3f}",
    "Imaginary part of |1> amplitude": "{:.3f}",
    "Probability of 0": "{:.3f}",
    "Probability of 1": "{:.3f}",
    "Bloch x": "{:.3f}",
    "Bloch y": "{:.3f}",
}))

## Example: phase encoding the scaled flower $(0.40,0.80)$

For scaled petal length,

$$
u_{\mathrm{length}}=0.40,
$$

so the phase angle is

$$
\phi_{\mathrm{length}}=2\pi(0.40)=0.80\pi.
$$

The petal-length qubit becomes

$$
|\phi_{\mathrm{length}}\rangle=\frac{|0\rangle+e^{i0.80\pi}|1\rangle}{\sqrt{2}}.
$$

For scaled petal width,

$$
u_{\mathrm{width}}=0.80,
$$

so the phase angle is

$$
\phi_{\mathrm{width}}=2\pi(0.80)=1.60\pi.
$$

The petal-width qubit becomes

$$
|\phi_{\mathrm{width}}\rangle=\frac{|0\rangle+e^{i1.60\pi}|1\rangle}{\sqrt{2}}.
$$

In both states, the magnitudes of the $|0\rangle$ and $|1\rangle$ amplitudes remain equal. Only their relative phases change.

## A single phase-encoded feature

The state

$$
|\phi_{\mathrm{phase}}(u)\rangle=\frac{|0\rangle+e^{i2\pi u}|1\rangle}{\sqrt{2}}
$$

has Bloch vector

$$
r(u)=\left(\cos(2\pi u),\sin(2\pi u),0\right).
$$

The $z$ coordinate is always zero. The state therefore remains on the equator of the Bloch sphere.

As $u$ increases from $0$ to $1$, the Bloch vector completes one full revolution around the equator:

$$
u=0\quad\longmapsto\quad|+\rangle,
$$

$$
u=\frac{1}{4}\quad\longmapsto\quad\frac{|0\rangle+i|1\rangle}{\sqrt{2}},
$$

$$
u=\frac{1}{2}\quad\longmapsto\quad|-\rangle,
$$

$$
u=\frac{3}{4}\quad\longmapsto\quad\frac{|0\rangle-i|1\rangle}{\sqrt{2}},
$$

$$
u=1\quad\longmapsto\quad|+\rangle.
$$

The beginning and end of the scaled interval are mapped to the same quantum state. This is the periodic identification introduced by our choice $\phi(u)=2\pi u$.

In [ ]:
GENERAL_PHASE_STEPS = 61

general_phase_scaled_values = np.linspace(0.0, 1.0, GENERAL_PHASE_STEPS)
general_phase_angles = 2.0 * np.pi * general_phase_scaled_values
general_phase_states = []

for phase_angle in general_phase_angles:
    circuit = QuantumCircuit(1)
    circuit.h(0)
    circuit.p(phase_angle, 0)
    general_phase_states.append(Statevector.from_instruction(circuit))

In [ ]:
def show_general_phase_step(step):
    """Display one state from the general phase-encoding trajectory.

    Args:
        step: Index of the state to display.
    """
    scaled_value = general_phase_scaled_values[step]
    phase_angle = general_phase_angles[step]
    state = general_phase_states[step]

    print(f"Scaled feature: u = {scaled_value:.3f}")
    print(f"Phase angle: phi = {phase_angle:.3f} radians = {phase_angle / np.pi:.3f} pi")
    print("Statevector:", np.round(state.data, decimals=4))

    bloch_figure = plot_bloch_multivector(state)
    display(bloch_figure)
    plt.close(bloch_figure)


general_phase_play = widgets.Play(value=0, min=0, max=GENERAL_PHASE_STEPS - 1, step=1, interval=100, description="Play")
general_phase_slider = widgets.IntSlider(value=0, min=0, max=GENERAL_PHASE_STEPS - 1, step=1, description="Position", continuous_update=False)

widgets.jslink((general_phase_play, "value"), (general_phase_slider, "value"))

general_phase_output = widgets.interactive_output(show_general_phase_step, {"step": general_phase_slider})

display(widgets.HBox([general_phase_play, general_phase_slider]), general_phase_output)

In [ ]:
scaled_values = np.linspace(0.0, 1.0, 500)
bloch_x_values = np.cos(2.0 * np.pi * scaled_values)
bloch_y_values = np.sin(2.0 * np.pi * scaled_values)
probability_zero = np.full_like(scaled_values, 0.5)
probability_one = np.full_like(scaled_values, 0.5)

fig, axes = plt.subplots(1, 2, figsize=(15, 5), constrained_layout=True)

axes[0].plot(scaled_values, bloch_x_values, label=r"$x(u)=\cos(2\pi u)$", linewidth=2.5, color="#2A6FDB")
axes[0].plot(scaled_values, bloch_y_values, label=r"$y(u)=\sin(2\pi u)$", linewidth=2.5, color="#E4572E")
axes[0].axhline(0.0, color="black", linewidth=0.8)
axes[0].set_xlabel("Scaled feature value")
axes[0].set_ylabel("Bloch-vector component")
axes[0].set_title("The feature changes the phase")
axes[0].legend()

axes[1].plot(scaled_values, probability_zero, label=r"$p(0\mid u)$", linewidth=2.5, color="#2A6FDB")
axes[1].plot(scaled_values, probability_one, label=r"$p(1\mid u)$", linewidth=2.5, linestyle="--", color="#E4572E")
axes[1].set_xlabel("Scaled feature value")
axes[1].set_ylabel("Measurement probability")
axes[1].set_ylim(0.0, 1.0)
axes[1].set_title("Computational-basis probabilities do not change")
axes[1].legend()

plt.show()

## The feature is invisible to direct computational-basis measurement

For every phase angle,

$$
|\phi_{\mathrm{phase}}(u)\rangle=\frac{|0\rangle+e^{i2\pi u}|1\rangle}{\sqrt{2}}.
$$

The computational-basis probabilities are

$$
p(0\mid u)=\frac{1}{2}
$$

and

$$
p(1\mid u)=\frac{1}{2}.
$$

They do not depend on $u$.

The data have not disappeared. They are stored in the relative phase between the two amplitudes. A direct computational-basis measurement cannot observe this relative phase.

The phase must first be converted into an amplitude difference through interference. For example, another Hadamard gate gives

$$
H|\phi_{\mathrm{phase}}(u)\rangle,
$$

after which the measurement probabilities become

$$
p_H(0\mid u)=\cos^2(\pi u)
$$

and

$$
p_H(1\mid u)=\sin^2(\pi u).
$$

A quantum kernel circuit or variational circuit creates the interference needed to access the encoded phase.

In [ ]:
phase_values = np.linspace(0.0, 1.0, 500)
probability_zero_before_hadamard = np.full_like(phase_values, 0.5)
probability_zero_after_hadamard = np.cos(np.pi * phase_values) ** 2

fig, ax = plt.subplots(figsize=(9, 5), constrained_layout=True)

ax.plot(phase_values, probability_zero_before_hadamard, label="Before final Hadamard", linewidth=2.5, color="#7A7A7A")
ax.plot(phase_values, probability_zero_after_hadamard, label="After final Hadamard", linewidth=2.5, color="#7B4AB5")

ax.set_xlabel("Scaled feature value")
ax.set_ylabel("Probability of measuring 0")
ax.set_ylim(0.0, 1.0)
ax.set_title("Interference converts relative phase into probability")
ax.legend()

plt.show()

## Phase encoding an actual Iris flower

We assign one qubit to each Iris feature:

| Qubit | Feature | Phase |
|---|---|---|
| $q_0$ | Petal length | $\phi_{\mathrm{length}}=2\pi u_{\mathrm{length}}$ |
| $q_1$ | Petal width | $\phi_{\mathrm{width}}=2\pi u_{\mathrm{width}}$ |

The circuit is

$$
U_{\mathrm{phase}}(u)=P^{(0)}(2\pi u_{\mathrm{length}})H^{(0)}P^{(1)}(2\pi u_{\mathrm{width}})H^{(1)}.
$$

Because the gates act independently on the two qubits, the encoded state is a product state.

Using Qiskit's displayed order $|q_1q_0\rangle$, the state is

$$
|\phi_{\mathrm{phase}}(u)\rangle=|\phi_{\mathrm{width}}\rangle_{q_1}\otimes|\phi_{\mathrm{length}}\rangle_{q_0}.
$$

In [ ]:
PHASE_SAMPLE_INDEX = 0

species_names = {0: "Versicolor", 1: "Virginica"}

phase_raw_flower = X_train_raw[PHASE_SAMPLE_INDEX]
phase_scaled_flower = X_train_unit[PHASE_SAMPLE_INDEX]
phase_species = species_names[y_train[PHASE_SAMPLE_INDEX]]

phase_flower_table = pd.DataFrame({
    "Feature": ["Petal length", "Petal width"],
    "Original measurement in cm": phase_raw_flower,
    "Scaled measurement": phase_scaled_flower,
})

print("Species:", phase_species)
display(phase_flower_table)

In [ ]:
actual_phases, actual_phase_steps = phase_coordinates(phase_scaled_flower, return_intermediates=True)

actual_phase_table = pd.DataFrame({
    "Feature": ["Petal length", "Petal width"],
    "Scaled value": actual_phase_steps["input_coordinates"][0],
    "Phase in radians": actual_phase_steps["phase_angles"][0],
    "Phase divided by pi": actual_phase_steps["phase_angles"][0] / np.pi,
    "Bloch x": actual_phase_steps["bloch_x"][0],
    "Bloch y": actual_phase_steps["bloch_y"][0],
})

display(actual_phase_table.style.format({
    "Scaled value": "{:.4f}",
    "Phase in radians": "{:.4f}",
    "Phase divided by pi": "{:.4f}",
    "Bloch x": "{:.4f}",
    "Bloch y": "{:.4f}",
}))

In [ ]:
actual_phase_circuit = QuantumCircuit(2, name="IrisPhaseEncoding")

actual_phase_circuit.h(0)
actual_phase_circuit.h(1)
actual_phase_circuit.p(actual_phases[0, 0], 0)
actual_phase_circuit.p(actual_phases[0, 1], 1)

actual_phase_state = Statevector.from_instruction(actual_phase_circuit)

display(actual_phase_circuit.draw(output="mpl", fold=-1))

print("Encoded statevector:", np.round(actual_phase_state.data, decimals=4))

In [ ]:
actual_phase_basis_states = ["|00>", "|01>", "|10>", "|11>"]
actual_phase_amplitudes = actual_phase_state.data
actual_phase_probabilities = np.abs(actual_phase_amplitudes) ** 2

formatted_phase_amplitudes = [f"{amplitude.real:.4f} {amplitude.imag:+.4f}i" for amplitude in actual_phase_amplitudes]

actual_phase_state_table = pd.DataFrame({
    "Basis state": actual_phase_basis_states,
    "Amplitude": formatted_phase_amplitudes,
    "Measurement probability": actual_phase_probabilities,
})

display(actual_phase_state_table.style.format({"Measurement probability": "{:.4f}"}))

print("Sum of probabilities:", actual_phase_probabilities.sum())

In [ ]:
actual_phase_basis_states = ["|00>", "|01>", "|10>", "|11>"]
actual_phase_amplitudes = actual_phase_state.data
actual_phase_probabilities = np.abs(actual_phase_amplitudes) ** 2

formatted_phase_amplitudes = [f"{amplitude.real:.4f} {amplitude.imag:+.4f}i" for amplitude in actual_phase_amplitudes]

actual_phase_state_table = pd.DataFrame({
    "Basis state": actual_phase_basis_states,
    "Amplitude": formatted_phase_amplitudes,
    "Measurement probability": actual_phase_probabilities,
})

display(actual_phase_state_table.style.format({"Measurement probability": "{:.4f}"}))

print("Sum of probabilities:", actual_phase_probabilities.sum())

In [ ]:
actual_phase_bloch_figure = plot_bloch_multivector(actual_phase_state)
display(actual_phase_bloch_figure)
plt.close(actual_phase_bloch_figure)

actual_phase_qsphere_figure = plot_state_qsphere(actual_phase_state)
display(actual_phase_qsphere_figure)
plt.close(actual_phase_qsphere_figure)

In [ ]:
actual_phase_density_matrix = DensityMatrix(actual_phase_state)

phase_reduced_state_qubit_0 = partial_trace(actual_phase_density_matrix, [1])
phase_reduced_state_qubit_1 = partial_trace(actual_phase_density_matrix, [0])

phase_state_diagnostics = pd.Series({
    "Global-state purity": float(np.real(purity(actual_phase_density_matrix))),
    "Qubit-0 purity": float(np.real(purity(phase_reduced_state_qubit_0))),
    "Qubit-1 purity": float(np.real(purity(phase_reduced_state_qubit_1))),
    "Two-qubit concurrence": float(concurrence(actual_phase_state)),
}, name="Value")

display(phase_state_diagnostics.to_frame().style.format(precision=6))

## Moving through the phase-encoding circuit

The phase-encoding circuit has two conceptual stages.

The Hadamard gates first create

$$
|00\rangle\longmapsto|++\rangle.
$$

The phase gates then move the two local Bloch vectors around their equators.

For the phase stage, introduce a progress parameter $t\in[0,1]$:

$$
|\psi(t)\rangle=P^{(0)}(t\phi_{\mathrm{length}})P^{(1)}(t\phi_{\mathrm{width}})|++\rangle.
$$

At $t=0$, the state is $|++\rangle$. At $t=1$, it is the complete phase encoding of the selected Iris flower.

In [ ]:
PHASE_ANIMATION_STEPS = 61

actual_phase_states = [Statevector.from_label("00")]
actual_phase_labels = [r"Initial state $|00\rangle$"]

hadamard_q0_circuit = QuantumCircuit(2)
hadamard_q0_circuit.h(0)
actual_phase_states.append(Statevector.from_instruction(hadamard_q0_circuit))
actual_phase_labels.append(r"Apply $H$ to $q_0$")

both_hadamards_circuit = QuantumCircuit(2)
both_hadamards_circuit.h(0)
both_hadamards_circuit.h(1)
actual_phase_states.append(Statevector.from_instruction(both_hadamards_circuit))
actual_phase_labels.append(r"Apply $H$ to $q_1$: the state is now $|++\rangle$")

phase_progress_values = np.linspace(0.0, 1.0, PHASE_ANIMATION_STEPS)[1:]

for progress in phase_progress_values:
    intermediate_circuit = QuantumCircuit(2)
    intermediate_circuit.h(0)
    intermediate_circuit.h(1)
    intermediate_circuit.p(progress * actual_phases[0, 0], 0)
    intermediate_circuit.p(progress * actual_phases[0, 1], 1)
    actual_phase_states.append(Statevector.from_instruction(intermediate_circuit))
    actual_phase_labels.append(f"Phase-encoding progress: t = {progress:.3f}")

In [ ]:
def show_actual_phase_step(step):
    """Display one state from the Iris phase-encoding trajectory.

    Args:
        step: Index of the trajectory state to display.
    """
    state = actual_phase_states[step]

    print(actual_phase_labels[step])
    print("Statevector:", np.round(state.data, decimals=4))

    bloch_figure = plot_bloch_multivector(state)
    display(bloch_figure)
    plt.close(bloch_figure)


actual_phase_play = widgets.Play(value=0, min=0, max=len(actual_phase_states) - 1, step=1, interval=100, description="Play")
actual_phase_slider = widgets.IntSlider(value=0, min=0, max=len(actual_phase_states) - 1, step=1, description="Progress", continuous_update=False)

widgets.jslink((actual_phase_play, "value"), (actual_phase_slider, "value"))

actual_phase_output = widgets.interactive_output(show_actual_phase_step, {"step": actual_phase_slider})

display(widgets.HBox([actual_phase_play, actual_phase_slider]), actual_phase_output)

## Interpreting the phase animation

The initial Hadamard gates move both qubits from the north poles of their Bloch spheres to the positive $x$ direction on the equator.

The phase gates then rotate the Bloch vectors around the $z$ axes.

Petal length controls the angular position of the first Bloch vector. Petal width controls the angular position of the second Bloch vector.

Both vectors remain on the surfaces of their spheres because the encoded state remains a pure product state. The phase-encoding layer does not create entanglement.

Unlike angle encoding, the motion is periodic. Completing a full $2\pi$ revolution returns a qubit to the same physical state.

# Entangling $ZZ$ encoding

Angle and phase encoding treat the two Iris features independently. Petal length controls one single-qubit gate, while petal width controls another.

The $ZZ$ feature map adds a data-dependent two-qubit interaction. The encoded state can therefore depend jointly on petal length and petal width.

Let

$$
u=(u_{\mathrm{length}},u_{\mathrm{width}})\in[0,1]^2
$$

denote the scaled Iris features. We first define

$$
x_0=\pi u_{\mathrm{length}}
$$

and

$$
x_1=\pi u_{\mathrm{width}}.
$$

For the Qiskit $ZZ$ feature map with scaling parameter $\alpha$, the single-feature phases are

$$
a_0(x)=\alpha x_0
$$

and

$$
a_1(x)=\alpha x_1.
$$

The two-feature interaction phase is

$$
\lambda(x)=\alpha(\pi-x_0)(\pi-x_1).
$$

Throughout this notebook, we use

$$
\alpha=2.
$$

The nonlinear product in $\lambda(x)$ is what distinguishes the $ZZ$ feature map from a product of independent phase encodings.

In [ ]:
ZZ_ALPHA = 2.0


def zz_coordinates(X, alpha=ZZ_ALPHA, return_intermediates=False):
    """Convert scaled Iris measurements into ZZ feature-map coordinates.

    The scaled petal measurements are mapped from ``[0, 1]`` into
    ``[0, pi]``. Qiskit's ZZ feature map then constructs single-qubit
    phases and a nonlinear two-qubit interaction phase.

    Args:
        X: Input data with shape ``(n_samples, 2)`` or ``(2,)``. Each row contains scaled petal length followed by scaled petal width.
        alpha: Multiplicative phase factor used by the ZZ feature map.
        return_intermediates: Whether to return the single-qubit and interaction phases.

    Returns:
        If ``return_intermediates`` is ``False``, returns the feature-map coordinates with shape ``(n_samples, 2)``.

        If ``return_intermediates`` is ``True``, returns a tuple containing the coordinates and a dictionary of intermediate arrays.

    Raises:
        ValueError: If the input does not contain exactly two features.
        ValueError: If any input value lies outside the interval ``[0, 1]``.
    """
    # Convert the input into a floating-point NumPy array.
    input_coordinates = np.asarray(X, dtype=float)

    # Convert one flower from shape (2,) into shape (1, 2).
    input_coordinates = np.atleast_2d(input_coordinates)

    # Verify that each flower has petal length and petal width.
    if input_coordinates.shape[1] != 2:
        raise ValueError("Expected two features: [scaled petal length, scaled petal width].")

    # Verify that the features have been scaled into the unit interval.
    if np.any(input_coordinates < 0.0) or np.any(input_coordinates > 1.0):
        raise ValueError("All input features must lie in the interval [0, 1].")

    # Map the scaled features from [0, 1] into [0, pi].
    feature_map_coordinates = np.pi * input_coordinates

    # Construct the two individual data-dependent phases.
    single_qubit_phases = alpha * feature_map_coordinates

    # Evaluate the nonlinear interaction feature (pi - x_0)(pi - x_1).
    interaction_features = (np.pi - feature_map_coordinates[:, 0]) * (np.pi - feature_map_coordinates[:, 1])

    # Multiply the interaction feature by alpha to obtain the interaction phase.
    interaction_phases = alpha * interaction_features

    # Return only the two feature-map coordinates during ordinary preprocessing.
    if not return_intermediates:
        return feature_map_coordinates

    # Return the complete calculation for demonstrations.
    intermediates = {
        "input_coordinates": input_coordinates,
        "feature_map_coordinates": feature_map_coordinates,
        "single_qubit_phases": single_qubit_phases,
        "interaction_features": interaction_features,
        "interaction_phases": interaction_phases,
        "alpha": alpha,
    }

    return feature_map_coordinates, intermediates

In [ ]:
example_flower = [0.40, 0.80]

example_zz_coordinates, zz_steps = zz_coordinates(example_flower, return_intermediates=True)

zz_trace = pd.DataFrame({
    "Feature": ["Petal length", "Petal width"],
    "Scaled value": zz_steps["input_coordinates"][0],
    "Feature-map coordinate": zz_steps["feature_map_coordinates"][0],
    "Coordinate divided by pi": zz_steps["feature_map_coordinates"][0] / np.pi,
    "Single-qubit phase": zz_steps["single_qubit_phases"][0],
    "Single-qubit phase divided by pi": zz_steps["single_qubit_phases"][0] / np.pi,
})

display(zz_trace.style.format({
    "Scaled value": "{:.4f}",
    "Feature-map coordinate": "{:.4f}",
    "Coordinate divided by pi": "{:.4f}",
    "Single-qubit phase": "{:.4f}",
    "Single-qubit phase divided by pi": "{:.4f}",
}))

print("Interaction feature:", zz_steps["interaction_features"][0])
print("Interaction phase:", zz_steps["interaction_phases"][0])
print("Interaction phase divided by pi:", zz_steps["interaction_phases"][0] / np.pi)

## Example: encoding the scaled flower $(0.40,0.80)$

The scaled petal measurements are

$$
u_{\mathrm{length}}=0.40
$$

and

$$
u_{\mathrm{width}}=0.80.
$$

The feature-map coordinates are

$$
x_0=\pi(0.40)=0.40\pi
$$

and

$$
x_1=\pi(0.80)=0.80\pi.
$$

Using $\alpha=2$, the individual phases are

$$
a_0=2x_0=0.80\pi
$$

and

$$
a_1=2x_1=1.60\pi.
$$

The interaction feature is

$$
\phi_{01}(x)=(\pi-x_0)(\pi-x_1).
$$

For this flower,

$$
\phi_{01}(x)=(0.60\pi)(0.20\pi)=0.12\pi^2.
$$

The interaction phase is therefore

$$
\lambda(x)=2(0.12\pi^2)=0.24\pi^2.
$$

The single-qubit phases encode the two features separately. The interaction phase encodes their joint value.

## The two-qubit $ZZ$ feature-map circuit

For one repetition, the Qiskit circuit performs the following operations:

- apply Hadamard gates to create $|++\rangle$;
- apply the phase $a_0(x)$ to $q_0$;
- apply the phase $a_1(x)$ to $q_1$;
- compute the parity of $q_0$ and $q_1$ with a controlled-$X$ gate;
- apply the interaction phase $\lambda(x)$;
- uncompute the parity.

The interaction block is

$$
\operatorname{CX}_{0,1}P_1(\lambda)\operatorname{CX}_{0,1}.
$$

Its matrix is

$$
\operatorname{diag}(1,e^{i\lambda},e^{i\lambda},1).
$$

Up to a global phase, this is a $ZZ$ rotation:

$$
\operatorname{CX}_{0,1}P_1(\lambda)\operatorname{CX}_{0,1}=e^{i\lambda/2}R_{ZZ}(\lambda).
$$

The global phase $e^{i\lambda/2}$ has no observable effect. The parity circuit therefore implements a genuine two-qubit $ZZ$ interaction.

In [ ]:
parameterized_zz_feature_map = zz_feature_map(feature_dimension=2, reps=1, entanglement="full", alpha=ZZ_ALPHA, insert_barriers=True)

display(parameterized_zz_feature_map.draw(output="mpl", fold=-1))
display(parameterized_zz_feature_map.decompose().draw(output="mpl", fold=-1))

## What does a $ZZ$ interaction do?

Before using the full data-dependent feature map, we isolate the central two-qubit operation:

$$
|\psi(\lambda)\rangle=R_{ZZ}(\lambda)|++\rangle.
$$

The initial state is

$$
|++\rangle=\frac{|00\rangle+|01\rangle+|10\rangle+|11\rangle}{2}.
$$

The $ZZ$ rotation is

$$
R_{ZZ}(\lambda)=\exp\left(-i\frac{\lambda}{2}Z\otimes Z\right).
$$

It applies one phase to the even-parity states $|00\rangle$ and $|11\rangle$ and the opposite phase to the odd-parity states $|01\rangle$ and $|10\rangle$.

The state becomes

$$
|\psi(\lambda)\rangle=\frac{e^{-i\lambda/2}|00\rangle+e^{i\lambda/2}|01\rangle+e^{i\lambda/2}|10\rangle+e^{-i\lambda/2}|11\rangle}{2}.
$$

The computational-basis probabilities remain equal to $1/4$. The interaction changes relative phases and entanglement, not computational-basis probabilities.

In [ ]:
GENERAL_ZZ_STEPS = 81

general_zz_angles = np.linspace(0.0, 2.0 * np.pi, GENERAL_ZZ_STEPS)
general_zz_states = []
general_zz_concurrences = []
general_zz_local_purities = []

for interaction_angle in general_zz_angles:
    circuit = QuantumCircuit(2)
    circuit.h(0)
    circuit.h(1)
    circuit.rzz(interaction_angle, 0, 1)

    state = Statevector.from_instruction(circuit)
    reduced_state = partial_trace(DensityMatrix(state), [1])

    general_zz_states.append(state)
    general_zz_concurrences.append(float(concurrence(state)))
    general_zz_local_purities.append(float(np.real(purity(reduced_state))))

In [ ]:
def show_general_zz_step(step):
    """Display one state from the general ZZ-interaction trajectory.

    Args:
        step: Index of the interaction state to display.
    """
    interaction_angle = general_zz_angles[step]
    state = general_zz_states[step]

    print(f"Interaction angle: lambda = {interaction_angle:.3f} radians = {interaction_angle / np.pi:.3f} pi")
    print(f"Concurrence: {general_zz_concurrences[step]:.4f}")
    print(f"Qubit-0 purity: {general_zz_local_purities[step]:.4f}")
    print("Computational-basis probabilities:", np.round(np.abs(state.data) ** 2, decimals=4))

    bloch_figure = plot_bloch_multivector(state)
    display(bloch_figure)
    plt.close(bloch_figure)

    qsphere_figure = plot_state_qsphere(state)
    display(qsphere_figure)
    plt.close(qsphere_figure)


general_zz_play = widgets.Play(value=0, min=0, max=GENERAL_ZZ_STEPS - 1, step=1, interval=100, description="Play")
general_zz_slider = widgets.IntSlider(value=0, min=0, max=GENERAL_ZZ_STEPS - 1, step=1, description="Interaction", continuous_update=False)

widgets.jslink((general_zz_play, "value"), (general_zz_slider, "value"))

general_zz_output = widgets.interactive_output(show_general_zz_step, {"step": general_zz_slider})

display(widgets.HBox([general_zz_play, general_zz_slider]), general_zz_output)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5), constrained_layout=True)

axes[0].plot(general_zz_angles / np.pi, general_zz_concurrences, linewidth=2.5, color="#E4572E")
axes[0].set_xlabel(r"Interaction angle $\lambda/\pi$")
axes[0].set_ylabel("Concurrence")
axes[0].set_ylim(-0.02, 1.02)
axes[0].set_title("Entanglement produced by the ZZ interaction")

axes[1].plot(general_zz_angles / np.pi, general_zz_local_purities, linewidth=2.5, color="#2A6FDB")
axes[1].set_xlabel(r"Interaction angle $\lambda/\pi$")
axes[1].set_ylabel("One-qubit purity")
axes[1].set_ylim(0.48, 1.02)
axes[1].set_title("Local purity decreases as entanglement grows")

plt.show()

## Entanglement generated by the interaction

For the state

$$
|\psi(\lambda)\rangle=R_{ZZ}(\lambda)|++\rangle,
$$

the concurrence is

$$
C(\lambda)=|\sin\lambda|.
$$

The state is separable when

$$
\lambda=k\pi
$$

for an integer $k$.

It is maximally entangled when

$$
\lambda=\frac{\pi}{2}+k\pi.
$$

The purity of either reduced one-qubit state is

$$
\operatorname{Tr}(\rho_0^2)=\operatorname{Tr}(\rho_1^2)=\frac{1+\cos^2\lambda}{2}.
$$

As entanglement grows, the local Bloch vectors move into their spheres. At maximal entanglement, both local Bloch vectors vanish even though the complete two-qubit state remains pure.

This is information stored entirely in correlations. It cannot be recovered from either individual qubit alone.

## Encoding an actual Iris flower

We now evaluate the complete $ZZ$ feature map on a real Iris observation.

The encoding has three data-dependent quantities:

$$
a_0=2\pi u_{\mathrm{length}},
$$

$$
a_1=2\pi u_{\mathrm{width}},
$$

and

$$
\lambda=2\left(\pi-\pi u_{\mathrm{length}}\right)\left(\pi-\pi u_{\mathrm{width}}\right).
$$

The individual phases determine the local positions of the qubits. The interaction phase determines how strongly the two feature coordinates become correlated in the encoded state.

In [ ]:
ZZ_SAMPLE_INDEX = 0

species_names = {0: "Versicolor", 1: "Virginica"}

zz_raw_flower = X_train_raw[ZZ_SAMPLE_INDEX]
zz_scaled_flower = X_train_unit[ZZ_SAMPLE_INDEX]
zz_species = species_names[y_train[ZZ_SAMPLE_INDEX]]

zz_flower_table = pd.DataFrame({
    "Feature": ["Petal length", "Petal width"],
    "Original measurement in cm": zz_raw_flower,
    "Scaled measurement": zz_scaled_flower,
})

print("Species:", zz_species)
display(zz_flower_table)

In [ ]:
actual_zz_coordinates, actual_zz_steps = zz_coordinates(zz_scaled_flower, return_intermediates=True)

actual_zz_table = pd.DataFrame({
    "Feature": ["Petal length", "Petal width"],
    "Scaled value": actual_zz_steps["input_coordinates"][0],
    "Feature-map coordinate": actual_zz_steps["feature_map_coordinates"][0],
    "Single-qubit phase": actual_zz_steps["single_qubit_phases"][0],
})

display(actual_zz_table.style.format({
    "Scaled value": "{:.4f}",
    "Feature-map coordinate": "{:.4f}",
    "Single-qubit phase": "{:.4f}",
}))

print("Interaction feature:", actual_zz_steps["interaction_features"][0])
print("Interaction phase:", actual_zz_steps["interaction_phases"][0])
print("Interaction phase divided by pi:", actual_zz_steps["interaction_phases"][0] / np.pi)

In [ ]:
actual_zz_circuit = parameterized_zz_feature_map.assign_parameters(actual_zz_coordinates[0])
actual_zz_state = Statevector.from_instruction(actual_zz_circuit)

display(actual_zz_circuit.decompose().draw(output="mpl", fold=-1))

print("Encoded statevector:", np.round(actual_zz_state.data, decimals=4))

In [ ]:
actual_zz_basis_states = ["|00>", "|01>", "|10>", "|11>"]
actual_zz_amplitudes = actual_zz_state.data
actual_zz_probabilities = np.abs(actual_zz_amplitudes) ** 2

formatted_zz_amplitudes = [f"{amplitude.real:.4f} {amplitude.imag:+.4f}i" for amplitude in actual_zz_amplitudes]

actual_zz_state_table = pd.DataFrame({
    "Basis state": actual_zz_basis_states,
    "Amplitude": formatted_zz_amplitudes,
    "Measurement probability": actual_zz_probabilities,
})

display(actual_zz_state_table.style.format({"Measurement probability": "{:.4f}"}))

print("Sum of probabilities:", actual_zz_probabilities.sum())

In [ ]:
actual_zz_bloch_figure = plot_bloch_multivector(actual_zz_state)
display(actual_zz_bloch_figure)
plt.close(actual_zz_bloch_figure)

actual_zz_qsphere_figure = plot_state_qsphere(actual_zz_state)
display(actual_zz_qsphere_figure)
plt.close(actual_zz_qsphere_figure)

## What should we see on the Bloch spheres?

The global $ZZ$-encoded state is pure:

$$
\operatorname{Tr}(\rho^2)=1.
$$

If the interaction produces entanglement, the reduced one-qubit states are mixed:

$$
\operatorname{Tr}(\rho_0^2)<1
$$

and

$$
\operatorname{Tr}(\rho_1^2)<1.
$$

Their Bloch vectors therefore lie inside their spheres.

This does not mean that information has been destroyed. The missing local information has moved into two-qubit correlations.

The Q-sphere shows the complete joint state, while the Bloch spheres show only the reduced states of the individual qubits.

In [ ]:
actual_single_qubit_phases = actual_zz_steps["single_qubit_phases"][0]
actual_interaction_phase = actual_zz_steps["interaction_phases"][0]

equivalent_zz_circuit = QuantumCircuit(2)

equivalent_zz_circuit.h(0)
equivalent_zz_circuit.h(1)
equivalent_zz_circuit.p(actual_single_qubit_phases[0], 0)
equivalent_zz_circuit.p(actual_single_qubit_phases[1], 1)
equivalent_zz_circuit.rzz(actual_interaction_phase, 0, 1)

equivalent_zz_state = Statevector.from_instruction(equivalent_zz_circuit)

equivalent_state_fidelity = np.abs(np.vdot(actual_zz_state.data, equivalent_zz_state.data)) ** 2

display(equivalent_zz_circuit.draw(output="mpl", fold=-1))
print("Fidelity with the Qiskit ZZ feature-map state:", equivalent_state_fidelity)

## Watching the interaction encode the feature pair

We first prepare the product state containing the two individual feature phases:

$$
|\psi_{\mathrm{local}}\rangle=P_0(a_0)P_1(a_1)|++\rangle.
$$

We then gradually increase the $ZZ$ interaction:

$$
|\psi(t)\rangle=R_{ZZ}(t\lambda)|\psi_{\mathrm{local}}\rangle,
$$

where $t\in[0,1]$.

At $t=0$, the state contains only independent single-feature phases.

At $t=1$, the state is equivalent to the complete Qiskit $ZZ$ feature-map state, up to a global phase.

This animation isolates exactly what the interaction adds to the encoding.

In [ ]:
ZZ_ANIMATION_STEPS = 61

zz_progress_values = np.linspace(0.0, 1.0, ZZ_ANIMATION_STEPS)
actual_zz_animation_states = []
actual_zz_animation_concurrences = []

for progress in zz_progress_values:
    intermediate_circuit = QuantumCircuit(2)
    intermediate_circuit.h(0)
    intermediate_circuit.h(1)
    intermediate_circuit.p(actual_single_qubit_phases[0], 0)
    intermediate_circuit.p(actual_single_qubit_phases[1], 1)
    intermediate_circuit.rzz(progress * actual_interaction_phase, 0, 1)

    state = Statevector.from_instruction(intermediate_circuit)

    actual_zz_animation_states.append(state)
    actual_zz_animation_concurrences.append(float(concurrence(state)))

In [ ]:
def show_actual_zz_step(step):
    """Display one state from the Iris ZZ-encoding trajectory.

    Args:
        step: Index of the trajectory state to display.
    """
    progress = zz_progress_values[step]
    interaction_angle = progress * actual_interaction_phase
    state = actual_zz_animation_states[step]

    print(f"Encoding progress: t = {progress:.3f}")
    print(f"Current interaction phase: {interaction_angle:.3f} radians")
    print(f"Concurrence: {actual_zz_animation_concurrences[step]:.4f}")
    print("Computational-basis probabilities:", np.round(np.abs(state.data) ** 2, decimals=4))

    bloch_figure = plot_bloch_multivector(state)
    display(bloch_figure)
    plt.close(bloch_figure)

    qsphere_figure = plot_state_qsphere(state)
    display(qsphere_figure)
    plt.close(qsphere_figure)


actual_zz_play = widgets.Play(value=0, min=0, max=ZZ_ANIMATION_STEPS - 1, step=1, interval=100, description="Play")
actual_zz_slider = widgets.IntSlider(value=0, min=0, max=ZZ_ANIMATION_STEPS - 1, step=1, description="Interaction", continuous_update=False)

widgets.jslink((actual_zz_play, "value"), (actual_zz_slider, "value"))

actual_zz_output = widgets.interactive_output(show_actual_zz_step, {"step": actual_zz_slider})

display(widgets.HBox([actual_zz_play, actual_zz_slider]), actual_zz_output)

## Interpreting the moving $ZZ$ encoding

At the beginning of the animation, both qubits are in pure equatorial states determined by their individual feature phases. The state is separable, and both Bloch vectors lie on their sphere surfaces.

As the $ZZ$ interaction grows, the state generally becomes entangled. The individual Bloch vectors move into their spheres even though the complete state remains pure.

The computational-basis probabilities remain equal to $1/4$ throughout this process. The changing information is stored in phases and correlations rather than population differences.

The final state contains three kinds of data dependence:

- a phase determined by petal length;
- a phase determined by petal width;
- an entangling phase determined jointly by petal length and petal width.

## Which states can the $ZZ$ encoding reach?

A general two-qubit pure state requires six independent real parameters after removing normalization and global phase.

The Iris $ZZ$ encoding depends on only two classical variables:

$$
u_{\mathrm{length}}
$$

and

$$
u_{\mathrm{width}}.
$$

The single-qubit and interaction phases are all fixed functions of these two variables. The feature map therefore reaches only a two-dimensional surface inside the full two-qubit state space.

This surface is richer than the angle- or phase-encoding surfaces because it contains entangled states. However, it still cannot reach arbitrary two-qubit states.

The presence of entanglement does not mean that the encoding is universally expressive. It means that the reachable surface includes correlations unavailable to product-state encodings.

## The kernel induced by the $ZZ$ feature map

For one repetition, the encoded state has four equal-magnitude amplitudes with data-dependent phases.

Define

$$
a_0(x)=\alpha x_0,
$$

$$
a_1(x)=\alpha x_1,
$$

and

$$
\lambda(x)=\alpha(\pi-x_0)(\pi-x_1).
$$

Up to a global phase, the state has the form

$$
|\phi_{ZZ}(x)\rangle=\frac{1}{2}\left(|00\rangle+e^{i[a_0(x)+\lambda(x)]}|01\rangle+e^{i[a_1(x)+\lambda(x)]}|10\rangle+e^{i[a_0(x)+a_1(x)]}|11\rangle\right).
$$

For two inputs $x$ and $y$, define

$$
\Delta a_j=a_j(y)-a_j(x)
$$

and

$$
\Delta\lambda=\lambda(y)-\lambda(x).
$$

Their overlap is

$$
\langle\phi_{ZZ}(x)|\phi_{ZZ}(y)\rangle=\frac{1}{4}\left(1+e^{i(\Delta a_0+\Delta\lambda)}+e^{i(\Delta a_1+\Delta\lambda)}+e^{i(\Delta a_0+\Delta a_1)}\right).
$$

The fidelity kernel is

$$
K_{ZZ}(x,y)=\left|\langle\phi_{ZZ}(x)|\phi_{ZZ}(y)\rangle\right|^2.
$$

Unlike the angle and phase kernels, this expression does not factorize into a petal-length kernel multiplied by a petal-width kernel. The term $\Delta\lambda$ couples the two feature coordinates.

In [ ]:
zz_quantum_kernel = FidelityStatevectorKernel(feature_map=parameterized_zz_feature_map, auto_clear_cache=False, enforce_psd=True)

In [ ]:
ZZ_KERNEL_GRID_SIZE = 100

zz_kernel_axis = np.linspace(0.0, 1.0, ZZ_KERNEL_GRID_SIZE)
zz_kernel_length, zz_kernel_width = np.meshgrid(zz_kernel_axis, zz_kernel_axis)
zz_kernel_grid = np.column_stack([zz_kernel_length.ravel(), zz_kernel_width.ravel()])

zz_reference_flower = np.array([[0.50, 0.50]])

encoded_zz_grid = zz_coordinates(zz_kernel_grid)
encoded_zz_reference = zz_coordinates(zz_reference_flower)

zz_kernel_values = zz_quantum_kernel.evaluate(encoded_zz_grid, encoded_zz_reference)
zz_kernel_surface = zz_kernel_values.reshape(zz_kernel_length.shape)

fig, ax = plt.subplots(figsize=(8, 6), constrained_layout=True)

contour = ax.contourf(zz_kernel_length, zz_kernel_width, zz_kernel_surface, levels=np.linspace(0.0, 1.0, 21), cmap="viridis")

ax.scatter(zz_reference_flower[0, 0], zz_reference_flower[0, 1], marker="*", s=220, color="white", edgecolor="black", label="Reference flower")
ax.set_xlabel("Scaled petal length")
ax.set_ylabel("Scaled petal width")
ax.set_title("ZZ-kernel similarity to the reference point")
ax.legend()

fig.colorbar(contour, ax=ax, label=r"$K_{ZZ}(u,u_{\mathrm{ref}})$")

plt.show()

In [ ]:
encoded_zz_training_data = zz_coordinates(X_train_quantum)
zz_kernel_matrix = zz_quantum_kernel.evaluate(encoded_zz_training_data)

label_order = np.argsort(y_train_quantum)
ordered_zz_kernel = zz_kernel_matrix[np.ix_(label_order, label_order)]

fig, ax = plt.subplots(figsize=(7, 6), constrained_layout=True)

sns.heatmap(ordered_zz_kernel, vmin=0.0, vmax=1.0, cmap="mako", square=True, xticklabels=False, yticklabels=False, ax=ax)

ax.set_xlabel("Training flowers sorted by species")
ax.set_ylabel("Training flowers sorted by species")
ax.set_title("ZZ-encoding fidelity kernel")

plt.show()

## $ZZ$ encoding compared with phase encoding

Both feature maps begin by preparing $|++\rangle$ and applying data-dependent phases.

Phase encoding uses only local gates:

$$
U_{\mathrm{phase}}(u)=P_0(a_0)P_1(a_1)H^{\otimes2}.
$$

Its state remains separable, and its kernel factorizes between the two features.

The $ZZ$ map adds the interaction

$$
R_{ZZ}(\lambda(u)).
$$

Its state can become entangled, and its kernel generally does not factorize.

| Property | Phase encoding | $ZZ$ encoding |
|---|---|---|
| Qubits | 2 | 2 |
| Local feature phases | Yes | Yes |
| Joint feature phase | No | Yes |
| Entanglement in the feature map | No | Generally yes |
| Factorized kernel | Yes | Generally no |
| Two-qubit gates | None | Required |
| Direct computational-basis probabilities | Uniform | Uniform |
| Information location | Local phases | Local phases and correlations |

The $ZZ$ feature map makes joint feature dependence reachable, but it also increases two-qubit gate cost and can introduce rapidly varying phase structure. More entanglement and a more complicated kernel do not automatically improve classification.

# Comparing the encodings as classifiers

We now hold the Iris dataset and train/test split fixed while changing only the encoding.

We conduct two quantum-classification experiments.

### Quantum-kernel classifier

Each feature map defines the fidelity kernel

$$
K_\phi(u,v)=|\langle\phi(u)|\phi(v)\rangle|^2.
$$

A support-vector classifier learns

$$
f_{\mathrm{kernel}}(u)=\sum_{i\in\mathrm{SV}}\alpha_i y_iK_\phi(u_i,u)+b.
$$

This experiment directly tests whether the geometry created by an encoding is useful for separating Versicolor from Virginica.

### Variational quantum classifier

Each encoded state is followed by the same ansatz family:

$$
|\psi(u,\theta)\rangle=V(\theta)U_\phi(u)|0\rangle^{\otimes n}.
$$

The VQC learns a probability margin

$$
f_{\mathrm{VQC}}(u)=p_\theta(1\mid u)-p_\theta(0\mid u).
$$

This experiment tests both the encoding and the ability of the chosen variational circuit and optimizer to exploit it.

### Controls

Every encoding uses:

- the same training observations;
- the same test observations;
- exact statevector simulation;
- the same SVM regularization parameter;
- the same VQC ansatz family;
- the same VQC optimization budget;
- the same number of optimizer restarts.

The encodings require different numbers of qubits. Consequently, their VQC ansätze have different numbers of parameters. These resource differences are part of the practical encoding comparison.

In [ ]:
from time import perf_counter

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from qiskit import QuantumCircuit
from qiskit.circuit import ParameterVector
from qiskit.circuit.library import efficient_su2, zz_feature_map

from qiskit_machine_learning.algorithms import QSVC, VQC
from qiskit_machine_learning.circuit.library import raw_feature_vector
from qiskit_machine_learning.kernels import FidelityStatevectorKernel
from qiskit_machine_learning.optimizers import COBYLA
from qiskit_machine_learning.primitives import QMLSampler
from qiskit_machine_learning.utils import algorithm_globals

from sklearn.base import clone
from sklearn.inspection import DecisionBoundaryDisplay
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import ConfusionMatrixDisplay, accuracy_score, balanced_accuracy_score, f1_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, StandardScaler
from sklearn.svm import SVC

In [ ]:
SEED = 17
N_VQC_RESTARTS = 3
VQC_MAXITER = 100
VQC_ANSATZ_REPS = 1
SVM_REGULARIZATION = 1.0

rng = np.random.default_rng(SEED)
algorithm_globals.random_seed = SEED

## Dataset supplied to the classifiers

The classifiers receive the scaled petal measurements

$$
u=(u_{\mathrm{length}},u_{\mathrm{width}})\in[0,1]^2.
$$

The training arrays are

- `X_train_quantum`;
- `y_train_quantum`.

The held-out test arrays are

- `X_test_unit`;
- `y_test`.

The transformations `basis_coordinates`, `amplitude_coordinates`, `angle_coordinates`, `phase_coordinates`, and `zz_coordinates` map the same two-dimensional input into the coordinates required by each circuit.

In [ ]:
def make_basis_feature_map():
    """Construct the four-qubit basis feature map.

    Returns:
        A parameterized four-qubit circuit that prepares a computational-basis state when its parameters are binary.
    """
    bits = ParameterVector("b", 4)
    circuit = QuantumCircuit(4, name="Basis")

    for qubit, bit in enumerate(bits):
        circuit.ry(np.pi * bit, qubit)

    return circuit


def make_angle_feature_map():
    """Construct the two-qubit angle feature map.

    Returns:
        A parameterized circuit containing one Y rotation per Iris feature.
    """
    angles = ParameterVector("angle", 2)
    circuit = QuantumCircuit(2, name="Angle")

    circuit.ry(angles[0], 0)
    circuit.ry(angles[1], 1)

    return circuit


def make_phase_feature_map():
    """Construct the two-qubit phase feature map.

    Returns:
        A parameterized circuit containing one phase-encoded qubit per Iris feature.
    """
    phases = ParameterVector("phase", 2)
    circuit = QuantumCircuit(2, name="Phase")

    circuit.h(0)
    circuit.h(1)
    circuit.p(phases[0], 0)
    circuit.p(phases[1], 1)

    return circuit

In [ ]:
encoding_functions = {
    "Basis": basis_coordinates,
    "Amplitude": amplitude_coordinates,
    "Angle": angle_coordinates,
    "Phase": phase_coordinates,
    "ZZ": zz_coordinates,
}

encoding_transformers = {
    "Basis": FunctionTransformer(basis_coordinates, kw_args={"bins": N_BASIS_BINS}, validate=False),
    "Amplitude": FunctionTransformer(amplitude_coordinates, validate=False),
    "Angle": FunctionTransformer(angle_coordinates, validate=False),
    "Phase": FunctionTransformer(phase_coordinates, validate=False),
    "ZZ": FunctionTransformer(zz_coordinates, validate=False),
}

feature_maps = {
    "Basis": make_basis_feature_map(),
    "Amplitude": raw_feature_vector(4),
    "Angle": make_angle_feature_map(),
    "Phase": make_phase_feature_map(),
    "ZZ": zz_feature_map(feature_dimension=2, reps=1, entanglement="full", alpha=ZZ_ALPHA),
}

encoding_summary = pd.DataFrame({
    "Encoding": list(feature_maps),
    "Input coordinates": [feature_map.num_parameters for feature_map in feature_maps.values()],
    "Qubits": [feature_map.num_qubits for feature_map in feature_maps.values()],
}).set_index("Encoding")

display(encoding_summary)

Verify that everything is in order!


In [ ]:
encoded_training_sets = {}
encoded_test_sets = {}

for encoding_name, transformer in encoding_transformers.items():
    transformer.fit(X_train_quantum)
    encoded_training_sets[encoding_name] = transformer.transform(X_train_quantum)
    encoded_test_sets[encoding_name] = transformer.transform(X_test_unit)

transformation_summary = pd.DataFrame({
    "Encoding": list(encoded_training_sets),
    "Training shape": [encoded_training_sets[name].shape for name in encoded_training_sets],
    "Test shape": [encoded_test_sets[name].shape for name in encoded_test_sets],
}).set_index("Encoding")

display(transformation_summary)

# Quantum-kernel experiment

The kernel experiment is the cleanest test of the encoding itself.

For each encoding, the quantum computer or statevector simulator calculates

$$
K_\phi(u,v)=|\langle\phi(u)|\phi(v)\rangle|^2.
$$

The same support-vector classifier is then trained using this kernel.

Because the SVM optimization is convex, the result does not depend on random parameter initialization. Performance differences primarily reflect differences in the geometry induced by the feature maps.

In [ ]:
quantum_kernels = {
    name: FidelityStatevectorKernel(feature_map=feature_map.copy(), auto_clear_cache=False, enforce_psd=True)
    for name, feature_map in feature_maps.items()
}

In [ ]:
kernel_matrices = {}

for encoding_name, quantum_kernel in quantum_kernels.items():
    kernel_matrices[encoding_name] = quantum_kernel.evaluate(encoded_training_sets[encoding_name])

In [ ]:
from matplotlib.cm import ScalarMappable
from matplotlib.colors import Normalize


label_order = np.argsort(y_train_quantum)

fig, axes = plt.subplots(2, 3, figsize=(16, 10), constrained_layout=True)
axes = axes.ravel()

for ax, (encoding_name, kernel_matrix) in zip(axes, kernel_matrices.items()):
    ordered_kernel = kernel_matrix[np.ix_(label_order, label_order)]

    sns.heatmap(
        ordered_kernel,
        vmin=0.0,
        vmax=1.0,
        cmap="mako",
        square=True,
        xticklabels=False,
        yticklabels=False,
        cbar=False,
        ax=ax,
    )

    ax.set_title(encoding_name)
    ax.set_xlabel("Samples sorted by species")
    ax.set_ylabel("Samples sorted by species")

for ax in axes[len(kernel_matrices):]:
    ax.set_visible(False)

color_scale = ScalarMappable(norm=Normalize(vmin=0.0, vmax=1.0), cmap="mako")
color_scale.set_array([])

colorbar = fig.colorbar(color_scale, ax=axes[:len(kernel_matrices)], shrink=0.82, pad=0.03)
colorbar.set_label(r"Fidelity $K_\phi(x_i,x_j)$", fontsize=12)
colorbar.set_ticks([0.0, 0.25, 0.50, 0.75, 1.0])

fig.suptitle("Fidelity kernels induced by the five encodings", fontsize=16)

plt.show()

In [ ]:
def calculate_kernel_diagnostics(kernel_matrix, labels, tolerance=1e-10):
    """Calculate empirical geometry diagnostics for a kernel matrix.

    Args:
        kernel_matrix: Square training Gram matrix.
        labels: Binary class labels encoded as zero and one.
        tolerance: Eigenvalue threshold used to determine numerical rank.

    Returns:
        A dictionary containing effective rank, target alignment, average similarities, and condition number.
    """
    kernel_matrix = np.asarray(kernel_matrix, dtype=float)
    signed_labels = 2.0 * np.asarray(labels, dtype=float) - 1.0
    kernel_matrix = 0.5 * (kernel_matrix + kernel_matrix.T)

    eigenvalues = np.clip(np.linalg.eigvalsh(kernel_matrix), 0.0, None)
    numerical_rank = int(np.sum(eigenvalues > tolerance))
    effective_rank = eigenvalues.sum() ** 2 / np.square(eigenvalues).sum()

    target_kernel = np.outer(signed_labels, signed_labels)
    target_alignment = np.sum(kernel_matrix * target_kernel) / (np.linalg.norm(kernel_matrix, "fro") * np.linalg.norm(target_kernel, "fro"))

    diagonal_mask = np.eye(len(labels), dtype=bool)
    same_class_mask = signed_labels[:, None] == signed_labels[None, :]
    different_class_mask = ~same_class_mask

    mean_within_class = kernel_matrix[same_class_mask & ~diagonal_mask].mean()
    mean_between_class = kernel_matrix[different_class_mask].mean()
    positive_eigenvalues = eigenvalues[eigenvalues > tolerance]

    condition_number = np.inf if numerical_rank < len(eigenvalues) else positive_eigenvalues.max() / positive_eigenvalues.min()

    return {
        "Numerical rank": numerical_rank,
        "Effective rank": effective_rank,
        "Target alignment": target_alignment,
        "Within-class fidelity": mean_within_class,
        "Between-class fidelity": mean_between_class,
        "Within minus between": mean_within_class - mean_between_class,
        "Condition number": condition_number,
    }

In [ ]:
kernel_diagnostic_rows = []

for encoding_name, kernel_matrix in kernel_matrices.items():
    diagnostics = calculate_kernel_diagnostics(kernel_matrix, y_train_quantum)
    kernel_diagnostic_rows.append({"Encoding": encoding_name, **diagnostics})

kernel_diagnostics_table = pd.DataFrame(kernel_diagnostic_rows).set_index("Encoding")

display(kernel_diagnostics_table.sort_values("Target alignment", ascending=False).style.format(precision=4))

In [ ]:
kernel_models = {}
kernel_performance_rows = []

for encoding_name in feature_maps:
    transformer = clone(encoding_transformers[encoding_name])
    classifier = QSVC(quantum_kernel=quantum_kernels[encoding_name], C=SVM_REGULARIZATION)
    pipeline = Pipeline([("encoding", transformer), ("classifier", classifier)])

    start_time = perf_counter()
    pipeline.fit(X_train_quantum, y_train_quantum)
    training_time = perf_counter() - start_time

    train_predictions = pipeline.predict(X_train_quantum)
    test_predictions = pipeline.predict(X_test_unit)

    kernel_models[encoding_name] = pipeline

    kernel_performance_rows.append({
        "Encoding": encoding_name,
        "Training accuracy": accuracy_score(y_train_quantum, train_predictions),
        "Test accuracy": accuracy_score(y_test, test_predictions),
        "Balanced test accuracy": balanced_accuracy_score(y_test, test_predictions),
        "Test F1": f1_score(y_test, test_predictions),
        "Support vectors": pipeline.named_steps["classifier"].n_support_.sum(),
        "Training time in seconds": training_time,
    })

kernel_performance = pd.DataFrame(kernel_performance_rows).set_index("Encoding").sort_values("Test accuracy", ascending=False)

display(kernel_performance.style.format({
    "Training accuracy": "{:.3f}",
    "Test accuracy": "{:.3f}",
    "Balanced test accuracy": "{:.3f}",
    "Test F1": "{:.3f}",
    "Training time in seconds": "{:.3f}",
}))

###  RBF and logistic regression for comparison

Here we have the RBF and logistic regression to see how they perform against the quantum classifier.

In [ ]:
classical_models = {
    "Logistic regression": Pipeline([("scaling", StandardScaler()), ("classifier", LogisticRegression(random_state=SEED))]),
    "Classical RBF SVM": Pipeline([("scaling", StandardScaler()), ("classifier", SVC(kernel="rbf", C=SVM_REGULARIZATION, gamma="scale"))]),
}

classical_performance_rows = []

for model_name, model in classical_models.items():
    start_time = perf_counter()
    model.fit(X_train_quantum, y_train_quantum)
    training_time = perf_counter() - start_time

    train_predictions = model.predict(X_train_quantum)
    test_predictions = model.predict(X_test_unit)

    classical_performance_rows.append({
        "Model": model_name,
        "Training accuracy": accuracy_score(y_train_quantum, train_predictions),
        "Test accuracy": accuracy_score(y_test, test_predictions),
        "Balanced test accuracy": balanced_accuracy_score(y_test, test_predictions),
        "Test F1": f1_score(y_test, test_predictions),
        "Training time in seconds": training_time,
    })

classical_performance = pd.DataFrame(classical_performance_rows).set_index("Model")

display(classical_performance.style.format({
    "Training accuracy": "{:.3f}",
    "Test accuracy": "{:.3f}",
    "Balanced test accuracy": "{:.3f}",
    "Test F1": "{:.3f}",
    "Training time in seconds": "{:.3f}",
}))

In [ ]:
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

fig, axes = plt.subplots(2, 3, figsize=(16, 10), constrained_layout=True)
axes = axes.ravel()

# Restrict the decision grid to the scaled feature domain [0, 1] × [0, 1].
plot_limits = np.array([[0.0, 0.0], [1.0, 1.0]])

for ax, (encoding_name, model) in zip(axes, kernel_models.items()):
    # Filled contours show the kernel SVM decision-function value.
    DecisionBoundaryDisplay.from_estimator(model, plot_limits, response_method="decision_function", plot_method="contourf", grid_resolution=100, eps=0.0, cmap="RdBu_r", alpha=0.35, ax=ax)

    # The zero contour is the actual classification boundary.
    DecisionBoundaryDisplay.from_estimator(model, plot_limits, response_method="decision_function", plot_method="contour", levels=[0.0], grid_resolution=100, eps=0.0, colors="black", linewidths=2.0, ax=ax)

    # Plot the training flowers over the decision surface.
    sns.scatterplot(x=X_train_quantum[:, 0], y=X_train_quantum[:, 1], hue=y_train_quantum, palette={0: "#2A6FDB", 1: "#E4572E"}, edgecolor="black", linewidth=0.5, s=55, legend=False, ax=ax)

    test_accuracy = kernel_performance.loc[encoding_name, "Test accuracy"]

    ax.set_xlim(0.0, 1.0)
    ax.set_ylim(0.0, 1.0)
    ax.set_xlabel("Scaled petal length")
    ax.set_ylabel("Scaled petal width")
    ax.set_title(f"{encoding_name}\nTest accuracy = {test_accuracy:.3f}")

# Hide any unused subplot.
for ax in axes[len(kernel_models):]:
    ax.set_visible(False)

# Create explanatory legend entries.
legend_handles = [
    Patch(facecolor=plt.get_cmap("RdBu_r")(0.15), alpha=0.35, label=r"Negative score: predicts versicolor ($y=0$)"),
    Patch(facecolor=plt.get_cmap("RdBu_r")(0.85), alpha=0.35, label=r"Positive score: predicts virginica ($y=1$)"),
    Line2D([0], [0], color="black", linewidth=2.0, label=r"Decision boundary: $f(x)=0$"),
    Line2D([0], [0], marker="o", linestyle="none", markerfacecolor="#2A6FDB", markeredgecolor="black", markersize=8, label="Versicolor training sample"),
    Line2D([0], [0], marker="o", linestyle="none", markerfacecolor="#E4572E", markeredgecolor="black", markersize=8, label="Virginica training sample"),
]

fig.legend(handles=legend_handles, loc="outside lower center", ncol=3, frameon=True)
fig.suptitle("Decision functions learned from the five quantum kernels", fontsize=16)

plt.show()

## Reading the decision-function plots

Each panel shows the classifier obtained from one quantum encoding.

The filled contours represent the kernel SVM decision function,

$$
f(x)
=
\sum_{i=1}^{m}
\alpha_i y_i K_\phi(x_i,x)
+
b.
$$

The background should be interpreted as follows:

- Blue regions have $f(x)<0$ and are classified as Iris versicolor.
- Red regions have $f(x)>0$ and are classified as Iris virginica.
- The solid black contour marks $f(x)=0$, which is the learned decision boundary.
- Stronger colors indicate points farther from the decision boundary according to the SVM decision score.

The colored circles are the quantum training samples. Their locations are identical in every panel. Only the encoding—and therefore the quantum kernel—changes.

Consequently, differences between these boundaries show how each encoding changes the similarity geometry available to the classifier. A smooth boundary indicates a kernel that varies smoothly across the input space. Repeated or rapidly changing regions indicate periodicity, aliasing, quantization, or strong nonlinear interactions introduced by the encoding.

Decision-function magnitudes should primarily be interpreted within each panel. Their absolute scales are not necessarily comparable between independently trained SVM models.


In [ ]:
from sklearn.metrics import accuracy_score
from sklearn.svm import SVC

# Train an RBF-kernel SVM using the same quantum training subset.
classical_rbf_model = SVC(kernel="rbf", C=1.0, gamma="scale")
classical_rbf_model.fit(X_train_quantum, y_train_quantum)

# Evaluate the classifier on the held-out Iris test set.
classical_rbf_predictions = classical_rbf_model.predict(X_test_unit)
classical_rbf_test_accuracy = accuracy_score(y_test, classical_rbf_predictions)

# Scikit-learn converts gamma="scale" into this numerical value during fitting.
classical_rbf_gamma = classical_rbf_model._gamma

print(f"RBF gamma: {classical_rbf_gamma:.4f}")
print(f"Classical RBF test accuracy: {classical_rbf_test_accuracy:.3f}")

In [ ]:
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

fig, ax = plt.subplots(figsize=(9, 7), constrained_layout=True)

# Restrict the evaluation grid to the scaled Iris feature domain.
plot_limits = np.array([[0.0, 0.0], [1.0, 1.0]])

# Plot filled contours of the signed SVM decision function.
decision_display = DecisionBoundaryDisplay.from_estimator(classical_rbf_model, plot_limits, response_method="decision_function", plot_method="contourf", grid_resolution=150, eps=0.0, cmap="RdBu_r", alpha=0.35, ax=ax)

# Plot the zero contour, where the predicted class changes.
DecisionBoundaryDisplay.from_estimator(classical_rbf_model, plot_limits, response_method="decision_function", plot_method="contour", levels=[0.0], grid_resolution=150, eps=0.0, colors="black", linewidths=2.0, ax=ax)

# Plot the same training flowers used for the quantum-kernel models.
sns.scatterplot(x=X_train_quantum[:, 0], y=X_train_quantum[:, 1], hue=y_train_quantum, palette={0: "#2A6FDB", 1: "#E4572E"}, edgecolor="black", linewidth=0.5, s=70, legend=False, ax=ax)

# Add a colorbar showing the numerical decision-function value.
colorbar = fig.colorbar(decision_display.surface_, ax=ax, pad=0.03)
colorbar.set_label(r"RBF-SVM decision score $f(x)$")

# Explain the decision regions, boundary, and sample markers.
legend_handles = [
    Patch(facecolor=plt.get_cmap("RdBu_r")(0.15), alpha=0.35, label=r"$f(x)<0$: predicts versicolor"),
    Patch(facecolor=plt.get_cmap("RdBu_r")(0.85), alpha=0.35, label=r"$f(x)>0$: predicts virginica"),
    Line2D([0], [0], color="black", linewidth=2.0, label=r"$f(x)=0$: decision boundary"),
    Line2D([0], [0], marker="o", linestyle="none", markerfacecolor="#2A6FDB", markeredgecolor="black", markersize=8, label="Versicolor training sample"),
    Line2D([0], [0], marker="o", linestyle="none", markerfacecolor="#E4572E", markeredgecolor="black", markersize=8, label="Virginica training sample"),
]

ax.legend(handles=legend_handles, loc="best", frameon=True)
ax.set_xlim(0.0, 1.0)
ax.set_ylim(0.0, 1.0)
ax.set_xlabel("Scaled petal length")
ax.set_ylabel("Scaled petal width")
ax.set_title(f"Classical RBF-kernel decision function\nTest accuracy = {classical_rbf_test_accuracy:.3f}")

plt.show()

## Interpreting the classical RBF boundary

The filled contours show the signed SVM decision function,

$$
f(x)=\sum_{i=1}^{m}\alpha_i y_i K_{\mathrm{RBF}}(x_i,x)+b.
$$

Blue regions have $f(x)<0$ and are classified as Iris versicolor. Red regions have $f(x)>0$ and are classified as Iris virginica. The black contour marks $f(x)=0$, where the classifier changes its prediction.

The RBF kernel depends only on Euclidean distance in the scaled input space. It therefore generally produces smooth and spatially localized decision boundaries.

This behavior provides a useful reference for the quantum encodings:

- Basis encoding can produce block-like boundaries because it quantizes the input.
- Angle encoding usually produces smooth but trigonometric similarity patterns.
- Phase encoding can produce repeated regions because of its periodicity.
- $ZZ$ encoding can generate more intricate boundaries through interaction-dependent phases.
- Amplitude encoding measures similarity between normalized augmented vectors.
- The RBF kernel produces smooth local similarity without first mapping the data into a quantum state.

The comparison is not intended to establish a quantum advantage. It shows how different kernel definitions change the family of separating functions available to the classifier.

## Actually training the VQC classifier

We had seen the results of the how the kernels act like before, but now the hard part is to actually use a Variational Quantum Classifier to learn these kernels.

In [ ]:
from tqdm.auto import tqdm

vqc_pipelines = {encoding_name: [] for encoding_name in feature_maps}
vqc_loss_histories = {}
vqc_performance_rows = []

# Each encoding is trained once for every restart.
total_vqc_runs = len(feature_maps) * N_VQC_RESTARTS
progress_bar = tqdm(total=total_vqc_runs, desc="Training VQC models", unit="model")

for encoding_index, (encoding_name, feature_map) in enumerate(feature_maps.items()):
    for restart in range(N_VQC_RESTARTS):
        restart_seed = SEED + 100 * encoding_index + restart
        ansatz = efficient_su2(num_qubits=feature_map.num_qubits, reps=VQC_ANSATZ_REPS, entanglement="linear")
        initial_rng = np.random.default_rng(restart_seed)
        initial_point = initial_rng.uniform(-np.pi, np.pi, size=ansatz.num_parameters)
        objective_history = []

        def record_objective(weights, objective_value):
            """Record the loss reported during VQC optimization.

            Args:
                weights: Current variational parameters.
                objective_value: Current value of the objective function.
            """
            objective_history.append(float(objective_value))

        classifier = VQC(sampler=QMLSampler(shots=None, seed=restart_seed), feature_map=feature_map.copy(), ansatz=ansatz, loss="cross_entropy", optimizer=COBYLA(maxiter=VQC_MAXITER), initial_point=initial_point, callback=record_objective)
        pipeline = Pipeline([("encoding", clone(encoding_transformers[encoding_name])), ("classifier", classifier)])

        start_time = perf_counter()
        pipeline.fit(X_train_quantum, y_train_quantum)
        training_time = perf_counter() - start_time

        # Retrieve the fitted encoding transformer and VQC from the pipeline.
        fitted_transformer = pipeline.steps[0][1]
        fitted_vqc = pipeline.steps[-1][1]

        # Transform the original Iris coordinates into the coordinates expected by this feature map.
        encoded_training_data = fitted_transformer.transform(X_train_quantum)
        encoded_test_data = fitted_transformer.transform(X_test_unit)

        # Call the fitted VQC directly to avoid scikit-learn's incompatible estimator-tag check.
        train_predictions = fitted_vqc.predict(encoded_training_data)
        test_predictions = fitted_vqc.predict(encoded_test_data)

        # Convert Qiskit's prediction output into one-dimensional integer arrays.
        train_predictions = np.asarray(train_predictions).reshape(-1).astype(int)
        test_predictions = np.asarray(test_predictions).reshape(-1).astype(int)

        # Calculate the performance measures for this run.
        final_loss = objective_history[-1] if objective_history else np.nan
        training_accuracy = accuracy_score(y_train_quantum, train_predictions)
        test_accuracy = accuracy_score(y_test, test_predictions)
        balanced_test_accuracy = balanced_accuracy_score(y_test, test_predictions)
        test_f1 = f1_score(y_test, test_predictions)

        # Store the fitted pipeline and the optimization history.
        vqc_pipelines[encoding_name].append(pipeline)
        vqc_loss_histories[(encoding_name, restart)] = objective_history

        # Store one row of numerical results.
        vqc_performance_rows.append({
            "Encoding": encoding_name,
            "Restart": restart,
            "Qubits": feature_map.num_qubits,
            "Trainable parameters": ansatz.num_parameters,
            "Objective evaluations": len(objective_history),
            "Final loss": final_loss,
            "Training accuracy": training_accuracy,
            "Test accuracy": test_accuracy,
            "Balanced test accuracy": balanced_test_accuracy,
            "Test F1": test_f1,
            "Training time in seconds": training_time,
        })

        # Update the progress bar after completing this encoding and restart.
        progress_bar.set_postfix(encoding=encoding_name, restart=f"{restart + 1}/{N_VQC_RESTARTS}", test_accuracy=f"{test_accuracy:.3f}")
        progress_bar.update(1)

progress_bar.close()

In [ ]:
vqc_performance = pd.DataFrame(vqc_performance_rows)

display(vqc_performance.sort_values(["Encoding", "Final loss"]).style.format({
    "Final loss": "{:.4f}",
    "Training accuracy": "{:.3f}",
    "Test accuracy": "{:.3f}",
    "Balanced test accuracy": "{:.3f}",
    "Test F1": "{:.3f}",
    "Training time in seconds": "{:.2f}",
}))

In [ ]:
vqc_performance = pd.DataFrame(vqc_performance_rows)

display(vqc_performance.sort_values(["Encoding", "Final loss"]).style.format({
    "Final loss": "{:.4f}",
    "Training accuracy": "{:.3f}",
    "Test accuracy": "{:.3f}",
    "Balanced test accuracy": "{:.3f}",
    "Test F1": "{:.3f}",
    "Training time in seconds": "{:.2f}",
}))

In [ ]:
vqc_aggregate = vqc_performance.groupby("Encoding").agg(
    Qubits=("Qubits", "first"),
    Trainable_parameters=("Trainable parameters", "first"),
    Mean_final_loss=("Final loss", "mean"),
    Standard_deviation_final_loss=("Final loss", "std"),
    Mean_training_accuracy=("Training accuracy", "mean"),
    Standard_deviation_training_accuracy=("Training accuracy", "std"),
    Mean_test_accuracy=("Test accuracy", "mean"),
    Standard_deviation_test_accuracy=("Test accuracy", "std"),
    Mean_balanced_test_accuracy=("Balanced test accuracy", "mean"),
    Mean_test_F1=("Test F1", "mean"),
    Mean_training_time=("Training time in seconds", "mean"),
).sort_values("Mean_test_accuracy", ascending=False)

display(vqc_aggregate.style.format(precision=3))

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10), constrained_layout=True)
axes = axes.ravel()

for ax, encoding_name in zip(axes, feature_maps):
    encoding_histories = [vqc_loss_histories[(encoding_name, restart)] for restart in range(N_VQC_RESTARTS)]

    for restart, history in enumerate(encoding_histories):
        ax.plot(np.arange(1, len(history) + 1), history, linewidth=1.6, alpha=0.75, label=f"Restart {restart}")

    ax.set_title(encoding_name)
    ax.set_xlabel("Objective-function evaluation")
    ax.set_ylabel("Cross-entropy loss")
    ax.legend()

for ax in axes[len(feature_maps):]:
    ax.set_visible(False)

fig.suptitle("VQC optimization histories", fontsize=16)

plt.show()

## Selecting one VQC for visualization

The test set must not be used to choose a model.

For each encoding, we select the restart with the smallest final training loss:

$$
r_\star=\operatorname*{arg\,min}_r\mathcal{L}_{\mathrm{train}}^{(r)}.
$$

The reported mean and standard deviation across all restarts remain the more honest measures of optimization stability. The selected model is used only to visualize one representative decision function.

In [ ]:
best_vqc_models = {}
selected_vqc_rows = []

for encoding_name, pipelines in vqc_pipelines.items():
    encoding_results = vqc_performance[vqc_performance["Encoding"] == encoding_name]
    best_row_index = encoding_results["Final loss"].idxmin()
    best_restart = int(vqc_performance.loc[best_row_index, "Restart"])
    best_vqc_models[encoding_name] = pipelines[best_restart]

    selected_vqc_rows.append({
        "Encoding": encoding_name,
        "Selected restart": best_restart,
        "Training loss": vqc_performance.loc[best_row_index, "Final loss"],
        "Training accuracy": vqc_performance.loc[best_row_index, "Training accuracy"],
        "Test accuracy": vqc_performance.loc[best_row_index, "Test accuracy"],
    })

selected_vqc_table = pd.DataFrame(selected_vqc_rows).set_index("Encoding")

display(selected_vqc_table.style.format(precision=4))

In [ ]:
def evaluate_vqc_margin(pipeline, X, batch_size=512):
    """Evaluate the continuous probability margin of a fitted VQC pipeline.

    Args:
        pipeline: Fitted pipeline containing an encoding transformer and VQC.
        X: Unencoded input data with shape ``(n_samples, 2)``.
        batch_size: Maximum number of observations evaluated in one batch.

    Returns:
        An array containing ``p(class 1) - p(class 0)`` for every input.
    """
    encoded_data = pipeline.named_steps["encoding"].transform(X)
    classifier = pipeline.named_steps["classifier"]
    margins = []

    for start in range(0, len(encoded_data), batch_size):
        batch = encoded_data[start:start + batch_size]
        probabilities = np.asarray(classifier.neural_network.forward(batch, classifier.weights)).reshape(len(batch), -1)
        margins.append(probabilities[:, 1] - probabilities[:, 0])

    return np.concatenate(margins)

In [ ]:
VQC_GRID_RESOLUTION = 50

petal_length_axis = np.linspace(0.0, 1.0, VQC_GRID_RESOLUTION)
petal_width_axis = np.linspace(0.0, 1.0, VQC_GRID_RESOLUTION)
grid_length, grid_width = np.meshgrid(petal_length_axis, petal_width_axis)
decision_grid = np.column_stack([grid_length.ravel(), grid_width.ravel()])

fig, axes = plt.subplots(2, 3, figsize=(16, 10), constrained_layout=True)
axes = axes.ravel()

for ax, (encoding_name, model) in zip(axes, best_vqc_models.items()):
    margin = evaluate_vqc_margin(model, decision_grid).reshape(grid_length.shape)

    ax.contourf(grid_length, grid_width, margin, levels=np.linspace(-1.0, 1.0, 21), cmap="RdBu_r", alpha=0.4, extend="both")
    ax.contour(grid_length, grid_width, margin, levels=[0.0], colors="black", linewidths=1.5)
    sns.scatterplot(x=X_train_quantum[:, 0], y=X_train_quantum[:, 1], hue=y_train_quantum, palette={0: "#2A6FDB", 1: "#E4572E"}, edgecolor="black", linewidth=0.5, s=55, legend=False, ax=ax)

    test_accuracy = selected_vqc_table.loc[encoding_name, "Test accuracy"]
    ax.set_title(f"{encoding_name}\ntest accuracy = {test_accuracy:.3f}")
    ax.set_xlabel("Scaled petal length")
    ax.set_ylabel("Scaled petal width")

for ax in axes[len(best_vqc_models):]:
    ax.set_visible(False)

fig.suptitle("Representative VQC decision functions", fontsize=16)

plt.show()

In [ ]:
performance_comparison = pd.DataFrame(index=feature_maps.keys())

performance_comparison["Kernel training accuracy"] = kernel_performance["Training accuracy"]
performance_comparison["Kernel test accuracy"] = kernel_performance["Test accuracy"]
performance_comparison["Kernel target alignment"] = kernel_diagnostics_table["Target alignment"]
performance_comparison["Kernel effective rank"] = kernel_diagnostics_table["Effective rank"]
performance_comparison["Mean VQC training accuracy"] = vqc_aggregate["Mean_training_accuracy"]
performance_comparison["Mean VQC test accuracy"] = vqc_aggregate["Mean_test_accuracy"]
performance_comparison["VQC test accuracy standard deviation"] = vqc_aggregate["Standard_deviation_test_accuracy"]
performance_comparison["VQC parameters"] = vqc_aggregate["Trainable_parameters"]
performance_comparison["Qubits"] = vqc_aggregate["Qubits"]

display(performance_comparison.sort_values("Kernel test accuracy", ascending=False).style.format(precision=3))

In [ ]:
accuracy_plot_data = performance_comparison[["Kernel test accuracy", "Mean VQC test accuracy"]].reset_index()
accuracy_plot_data = accuracy_plot_data.melt(id_vars="index", var_name="Classifier", value_name="Test accuracy")
accuracy_plot_data = accuracy_plot_data.rename(columns={"index": "Encoding"})

fig, ax = plt.subplots(figsize=(11, 6), constrained_layout=True)

sns.barplot(data=accuracy_plot_data, x="Encoding", y="Test accuracy", hue="Classifier", palette=["#2A6FDB", "#E4572E"], ax=ax)

for encoding_index, encoding_name in enumerate(performance_comparison.index):
    vqc_mean = performance_comparison.loc[encoding_name, "Mean VQC test accuracy"]
    vqc_std = performance_comparison.loc[encoding_name, "VQC test accuracy standard deviation"]
    ax.errorbar(encoding_index + 0.2, vqc_mean, yerr=vqc_std, color="black", capsize=4, linewidth=1.3)

ax.axhline(0.5, color="black", linestyle=":", label="Random binary accuracy")
ax.set_ylim(0.0, 1.05)
ax.set_ylabel("Held-out test accuracy")
ax.set_title("Kernel SVM versus variational quantum classifier")

plt.show()

In [ ]:
baseline_comparison = pd.concat([
    classical_performance[["Training accuracy", "Test accuracy"]],
    kernel_performance[["Training accuracy", "Test accuracy"]].rename(index=lambda name: f"{name} quantum kernel"),
])

display(baseline_comparison.sort_values("Test accuracy", ascending=False).style.format(precision=3))

### Confusion matrices

In [ ]:
fig, axes = plt.subplots(len(feature_maps), 2, figsize=(10, 4 * len(feature_maps)), constrained_layout=True)

for row, encoding_name in enumerate(feature_maps):
    # The quantum-kernel SVM supports the ordinary scikit-learn prediction interface.
    kernel_predictions = kernel_models[encoding_name].predict(X_test_unit)

    # Retrieve the selected fitted VQC pipeline.
    vqc_pipeline = best_vqc_models[encoding_name]

    # Apply the fitted encoding transformation manually.
    fitted_transformer = vqc_pipeline.named_steps["encoding"]
    encoded_test_data = fitted_transformer.transform(X_test_unit)

    # Call the fitted Qiskit VQC directly to bypass the scikit-learn tag check.
    fitted_vqc = vqc_pipeline.named_steps["classifier"]
    vqc_predictions = fitted_vqc.predict(encoded_test_data)
    vqc_predictions = np.asarray(vqc_predictions).reshape(-1).astype(int)

    # Plot the kernel-SVM confusion matrix.
    ConfusionMatrixDisplay.from_predictions(y_test, kernel_predictions, display_labels=["Versicolor", "Virginica"], cmap="Blues", colorbar=False, ax=axes[row, 0])

    # Plot the VQC confusion matrix.
    ConfusionMatrixDisplay.from_predictions(y_test, vqc_predictions, display_labels=["Versicolor", "Virginica"], cmap="Oranges", colorbar=False, ax=axes[row, 1])

    axes[row, 0].set_title(f"{encoding_name}: quantum-kernel SVM")
    axes[row, 1].set_title(f"{encoding_name}: selected VQC")

fig.suptitle("Test-set confusion matrices for the quantum models", fontsize=16)

plt.show()